# Völva 4.0: XRD Crystallite Size and Strain Analysis

## Purpose

Völva 4.0 is a three-cell Jupyter tool for crystallite size and microstrain analysis from powder X-ray diffraction data. It processes Bragg-Brentano geometry data (two-column 2θ/intensity format), applies instrumental broadening correction from a LaB₆ or similar line profile standard, and performs five complementary analysis methods. Each method output includes assumptions, limitations, quality checks and literature references, making the tool suitable for both research and teaching contexts.

## Theoretical Background

### Peak Broadening in Powder Diffraction

Observed XRD peak profiles contain contributions from at least three sources: (i) the instrumental profile, determined by the diffractometer optics; (ii) crystallite size broadening, arising from the finite extent of coherently diffracting domains; and (iii) microstrain broadening, arising from non-uniform lattice distortions such as dislocations, point defects and compositional gradients. The challenge in line profile analysis is to separate these contributions.

### Instrumental Correction

The instrumental contribution is measured using a line profile standard, typically NIST SRM 660c (LaB₆), which has crystallite sizes large enough that size broadening is negligible. Völva fits each standard peak with a pseudo-Voigt function (with optional Kα₁/Kα₂ doublet deconvolution) to extract the instrumental FWHM as a function of 2θ.

The angular dependence of the instrumental FWHM is described by the Caglioti function (Caglioti, Paoletti & Ricci, 1958):

$$\text{FWHM}^2 = U \tan^2\theta + V \tan\theta + W$$

where U captures high-angle broadening (principally Kα₁/Kα₂ splitting), V represents the focusing geometry (typically negative, producing a minimum in FWHM at mid-angles), and W is the angle-independent contribution from slit widths and source size. The Caglioti function provides a continuous, physically motivated instrumental correction at any 2θ position, including positions where no standard peak exists. If fewer than four standard peaks are available, or if the fit is unphysical (W < 0), Völva falls back to linear interpolation between standard peak positions.

Sample broadening is obtained by quadrature subtraction:

$$\beta_{\text{corr}} = \sqrt{\beta_{\text{obs}}^2 - \beta_{\text{inst}}^2}$$

This is strictly valid only for Gaussian profiles. For Lorentzian profiles, linear subtraction is appropriate. Since real XRD peaks are Voigt functions (convolutions of Gaussian and Lorentzian components), the quadrature assumption introduces a small systematic error that varies with the Lorentzian fraction of the profile.

### Method 1: Scherrer Analysis

The Scherrer equation (Scherrer, 1918; Patterson, 1939) relates the corrected integral breadth to the volume-averaged crystallite dimension along the diffraction vector:

$$D = \frac{K\lambda}{\beta\cos\theta}$$

where K is a shape-dependent constant (0.94 for FWHM of spherical crystallites), λ is the X-ray wavelength and β is the corrected FWHM in radians. The equation assumes that all broadening arises from finite crystallite size, with no contribution from microstrain. It gives a lower bound on crystallite size because any strain broadening present will be attributed to a smaller apparent size.

The Scherrer equation is valid only for crystallite sizes below approximately 100 to 200 nm, above which the sample broadening approaches the precision of the instrumental correction. The quantity D is a column length, not a particle diameter, and the distinction is significant for non-equiaxed crystallites.

### Method 2: Monshi-Scherrer Analysis

Monshi, Foroughi & Monshi (2012) proposed a modification in which the Scherrer equation is linearised by taking logarithms:

$$\ln\beta = \ln\left(\frac{K\lambda}{L}\right) + \ln\left(\frac{1}{\cos\theta}\right)$$

A plot of ln(β) against ln(1/cos θ) should yield a straight line with slope = 1 and intercept = ln(Kλ/L). The slope provides a built-in quality check: significant deviation from unity indicates that the Scherrer model is inadequate for the material, either because of anisotropic broadening or because strain contributes significantly to the line widths. A slope exceeding 1.5 should be considered a failure of the method.

### Method 3: Williamson-Hall Analysis (UDM)

The Williamson-Hall method (Williamson & Hall, 1953) separates size and strain contributions by assuming that total broadening is the linear sum of the two:

$$\beta\cos\theta = \frac{K\lambda}{D} + 4\varepsilon\sin\theta$$

A plot of βcos θ against 4sin θ should yield a straight line with slope ε (microstrain) and intercept Kλ/D. The linear sum assumption is valid only for purely Lorentzian profiles; since real peaks are Voigt functions, the method systematically overestimates strain and underestimates size. The Uniform Deformation Model (UDM) further assumes isotropic strain across all crystallographic directions. Poor linearity (R² < 0.5) indicates that this assumption is violated.

### Methods 4 and 5: Halder-Wagner and SSP

The Halder-Wagner method (Halder & Wagner, 1966) provides a more physical treatment by assuming Voigt profiles, in which size broadening is Lorentzian and strain broadening is Gaussian:

$$\left(\frac{\beta^*}{d^*}\right)^2 = \frac{K}{D}\cdot\frac{\beta^*}{d^{*2}} + (2\varepsilon)^2$$

where β* = βcos θ/λ and d* = 2sin θ/λ. This formulation gives greater weight to low- and mid-angle peaks, where data quality is typically better, and is less sensitive to high-angle noise than the Williamson-Hall method.

The Size-Strain Plot (SSP) method uses an equivalent formulation with different axis variables. As demonstrated by Khorsand Zak et al. (2025), the SSP and Halder-Wagner methods are analytically identical and should give the same crystallite size. Any difference in reported strain values arises from different weighting of data points in the linear regression, not from different physics.

### Population Analysis

When the coefficient of variation (CV) of Scherrer sizes exceeds 20%, Völva reports a population analysis: the sizes are split at the median into large-D and small-D groups, with separate statistics for each. A Pearson correlation between D and sin(θ) is calculated to distinguish between strain-dominated broadening (strong negative correlation, r < −0.7) and genuine (hkl)-dependent anisotropy (weak correlation, |r| < 0.4).

## Architecture

Völva 4.0 is structured as three Jupyter notebook cells:

| Cell | Contents | Purpose |
|------|----------|---------|
| 1 | `volva_engine` | All science: data loading, peak fitting, instrumental correction, five analysis methods, results formatting |
| 2 | `volva_plots` | All plotting: eight publication-quality figure types with a colorblind-safe palette |
| 3 | `volva_gui` | tkinter GUI: file loading, parameter controls, view switching, peak selection, export |

The separation ensures that all scientific calculations can be used programmatically without the GUI, and that the plotting code can be maintained independently of both the science and the interface.

## Analysis Methods Summary

| Method | Size | Strain | Profile Assumption | Key Diagnostic |
|--------|------|--------|--------------------|----------------|
| Scherrer | ✓ | ✗ | None (single peak) | CV of sizes |
| Monshi-Scherrer | ✓ | ✗ | None (regression) | Slope ≈ 1.0 |
| Williamson-Hall | ✓ | ✓ | Lorentzian (linear sum) | R² of W-H plot |
| Halder-Wagner | ✓ | ✓ | Voigt (Gaussian + Lorentzian) | R² of H-W plot |
| SSP | ✓ | ✓ | Voigt (equivalent to H-W) | R² (same as H-W) |

## Quick Start

**Cell execution order:** Run Cells 1, 2 and 3 in sequence. Cell 3 launches the GUI window.

**Step 1: Load data.** Click "Load Sample Data" and select a two-column 2θ/intensity file (.xy, .txt, .dat or .csv). The XRD pattern will appear with the SNIP background estimate. If a LaB₆ standard is available, load it with "Load Standard (LaB6)". The standard is processed automatically and the Caglioti instrumental profile is fitted. Without a standard, instrumental broadening is not corrected and all reported sizes are upper bounds.

**Step 2: Set parameters.** The default values are appropriate for most Aeris and laboratory Bragg-Brentano data. Parameters that may need adjusting:
- **Prominence** (default 0.01): fraction of maximum intensity used as the peak detection threshold. Increase to 0.03 or 0.05 if the detector is noisy and too many spurious peaks are detected.
- **Fit window max** (default 2.5°): ceiling for the adaptive fit window. The engine scales each window to 4x the estimated FWHM, capped at this value. Rarely needs changing.
- **Std min R²** (default 0.7): minimum R² to accept a standard peak fit. LaB₆ peaks are sharp and should fit well; lowering this below 0.7 is not recommended.
- **Options > Settings** provides access to sample-side parameters (min R², min SNR, deduplication separation, SNIP iterations). For weak, broad peaks at high angles, reducing min R² to 0.6 may recover useful reflections.

**Step 3: Detect and fit peaks.** Click the green "Detect & Fit Peaks" button. The tool will subtract the background, detect peaks, fit each with a pseudo-Voigt profile (with adaptive window sizing), apply the instrumental correction and display the results. The peak table on the left shows all accepted peaks with tick/cross toggles. Untick any peaks that are clearly unreliable (e.g. overlapping reflections, artefacts) and click "Re-run" to repeat the analysis with the reduced peak set.

**Step 4: Run analysis methods.** Click "Run All Methods" (blue button) to run all five methods at once, or use the individual method buttons (Scherrer, W-H, etc.) to run them one at a time. The Results view (green button when populated) shows the full output with method notes.

**Step 5: Inspect and export.** Use the view buttons along the bottom to switch between plot types. The Gallery view provides a quick visual check of all peak fits. Use File > Export to save peak data and summary tables as CSV, or File > Save Figure to export the current plot at 600 DPI.

**Interpreting the output.** Every method output includes an "Assumptions", "Limitations" and "Quality Check" section. The comparison summary includes interpretation guidance. The key diagnostics to check are: (i) the Scherrer CV — if above 20%, the material has anisotropic broadening; (ii) the Monshi slope — should be approximately 1.0; (iii) the Williamson-Hall R² — below 0.5 indicates the isotropic strain model is inadequate; and (iv) the Halder-Wagner R² — if this is also poor, no isotropic method will give a reliable single value, and the Scherrer population analysis (large-D and small-D groups) is the more informative result.

## References

- Caglioti, G., Paoletti, A. & Ricci, F.P. (1958). *Nucl. Instrum.* **3**, 223-228.
- Halder, N.C. & Wagner, C.N.J. (1966). *Acta Cryst.* **20**, 312-313.
- Khorsand Zak, A., Abd. Majid, W.H., Abrishami, M.E. & Yousefi, R. (2025). *J. Alloys Compd.*
- Monshi, A., Foroughi, M.R. & Monshi, M.R. (2012). *World J. Nano Sci. Eng.* **2**, 154-160.
- Patterson, A.L. (1939). *Phys. Rev.* **56**, 978-982.
- Scherrer, P. (1918). *Nachr. Ges. Wiss. Göttingen* **2**, 98-100.
- Williamson, G.K. & Hall, W.H. (1953). *Acta Metall.* **1**, 22-31.

In [ ]:
"""
Völva 4.0 — XRD Crystallite Size and Strain Analysis Engine
=============================================================

Core analysis engine. No GUI code — pure science.

Methods implemented:
    1. Scherrer (standard, with resolution-limit detection)
    2. Monshi-Scherrer (ln β vs ln(1/cos θ), Monshi et al. 2012)
    3. Williamson-Hall UDM (β cos θ vs 4 sin θ)
    4. Halder-Wagner (Voigt approximation, correct H-W formulation)
    5. Size-Strain Plot (SSP, analytically equivalent to H-W)

Units convention throughout:
    - 2θ: degrees
    - β (FWHM): degrees internally, converted to radians for calculations
    - λ: nanometres
    - D (crystallite size): nanometres
    - ε (microstrain): dimensionless (reported as percentage where noted)

Author: N. Reeves-McLaren, University of Sheffield
"""

import numpy as np
from scipy.signal import find_peaks, savgol_filter
from scipy.ndimage import minimum_filter1d
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any
import logging
import warnings

try:
    from lmfit import models, Parameters, Model
    from lmfit.models import PseudoVoigtModel
    LMFIT_AVAILABLE = True
except ImportError:
    LMFIT_AVAILABLE = False
    warnings.warn("lmfit not available. Peak fitting will not function.")


# =============================================================================
# LOGGING
# =============================================================================

logger = logging.getLogger("volva")
logger.setLevel(logging.INFO)


# =============================================================================
# CONSTANTS
# =============================================================================

# Wavelength data: ka1 (nm), ka2 (nm), ka2/ka1 intensity ratio
WAVELENGTHS = {
    "Cu Ka": {"ka1": 0.154056, "ka2": 0.154439, "ratio": 0.5},
    "Mo Ka": {"ka1": 0.070930, "ka2": 0.071073, "ratio": 0.5},
    "Co Ka": {"ka1": 0.178897, "ka2": 0.179285, "ratio": 0.5},
    "Ag Ka": {"ka1": 0.055941, "ka2": 0.056380, "ratio": 0.5},
    "Cr Ka": {"ka1": 0.228970, "ka2": 0.229361, "ratio": 0.5},
}

SCHERRER_K = {
    "Spherical (FWHM)": 0.94,
    "General/Approximation": 0.9,
    "Spherical (Integral Breadth)": 0.89,
    "Cubic": 1.0,
    "Rod-like": 1.84,
    "Plate-like": 0.89,
}

# Broadening ratio below which a peak is considered resolution-limited.
# If FWHM_obs / FWHM_inst < this threshold, the sample broadening cannot
# be reliably separated from instrumental broadening.
RESOLUTION_LIMIT_RATIO = 1.2

# Minimum number of data points in a fitting window for a reliable fit
MIN_FIT_POINTS = 10

# Default maximum fitting window in degrees 2θ (adaptive window ceiling)
DEFAULT_FIT_WINDOW_DEG = 2.5

# --- Tuneable defaults (overridable via GUI Settings) ---
DEFAULT_MIN_R_SQUARED = 0.65      # Min R² to accept a peak fit
DEFAULT_MIN_SNR = 5.0             # Min SNR to attempt fitting
DEFAULT_SAMPLE_DEDUP_DEG = 0.3    # Sample peak dedup separation (°)
DEFAULT_STD_DEDUP_DEG = 0.1       # Standard peak dedup separation (°)
DEFAULT_SNIP_ITERATIONS = 40      # SNIP background iterations


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class PeakFitResult:
    """Result of fitting a single XRD peak."""
    success: bool
    center_deg: float = 0.0          # Fitted peak centre (2θ, degrees)
    fwhm_deg: float = 0.0           # FWHM of Kα1 component (degrees)
    amplitude: float = 0.0          # Peak amplitude (counts)
    fraction: float = 0.0           # Lorentzian fraction (0 = pure Gaussian, 1 = pure Lorentzian)
    sigma: float = 0.0              # Sigma parameter from fit
    r_squared: float = 0.0         # R² of fit
    reduced_chi_sq: float = 0.0    # Reduced χ²
    fit_x: Optional[np.ndarray] = None   # x-values for plotting fitted curve
    fit_y: Optional[np.ndarray] = None   # y-values for plotting fitted curve
    residuals: Optional[np.ndarray] = None  # Fit residuals
    data_x: Optional[np.ndarray] = None    # Observed data x in fit window
    data_y: Optional[np.ndarray] = None    # Observed data y in fit window
    doublet_fitted: bool = False    # Whether Kα1-Kα2 doublet was fitted
    message: str = ""


@dataclass
class AnalysedPeak:
    """A fully processed peak ready for size/strain analysis."""
    two_theta_deg: float            # Fitted 2θ position (degrees)
    theta_rad: float                # θ in radians (= two_theta_deg / 2, converted)
    fwhm_obs_deg: float             # Observed FWHM (degrees)
    fwhm_inst_deg: float            # Instrumental FWHM at this 2θ (degrees)
    beta_corrected_deg: float       # Corrected FWHM after deconvolution (degrees)
    beta_corrected_rad: float       # Corrected FWHM in radians
    resolution_limited: bool        # True if broadening ratio < threshold
    broadening_ratio: float         # FWHM_obs / FWHM_inst
    fit_quality: float              # R² of the peak fit
    d_spacing_nm: float             # d-spacing (nm) from Bragg's law
    intensity: float                # Peak amplitude


@dataclass
class LinearFitResult:
    """Result of a linear regression for W-H, H-W, SSP, or Monshi-Scherrer plots."""
    slope: float = 0.0
    intercept: float = 0.0
    r_squared: float = 0.0
    x_data: Optional[np.ndarray] = None
    y_data: Optional[np.ndarray] = None
    n_points: int = 0


# =============================================================================
# FILE LOADING
# =============================================================================

class XRDLoader:
    """Load XRD data from common file formats."""

    @staticmethod
    def load(file_path: str) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], str]:
        """
        Load two-column XRD data (2θ, intensity).

        Tries multiple delimiters. Validates and sorts by 2θ.

        Returns:
            (two_theta, intensity, message) or (None, None, error_message)
        """
        import os
        if not os.path.isfile(file_path):
            return None, None, f"File not found: {file_path}"

        for delimiter in [None, '\t', ',', ';', ' ']:
            try:
                data = np.loadtxt(file_path, delimiter=delimiter, comments='#')
                if data.ndim == 2 and data.shape[1] >= 2 and data.shape[0] >= MIN_FIT_POINTS:
                    two_theta = data[:, 0]
                    intensity = data[:, 1]

                    # Basic validation
                    if np.any(two_theta < 0) or np.any(two_theta > 180):
                        continue

                    # Sort by 2θ
                    sort_idx = np.argsort(two_theta)
                    two_theta = two_theta[sort_idx]
                    intensity = intensity[sort_idx]

                    # Floor negative intensities to zero
                    intensity = np.maximum(intensity, 0)

                    return two_theta, intensity, f"Loaded {len(two_theta)} points from {os.path.basename(file_path)}"
            except Exception:
                continue

        return None, None, f"Could not parse: {file_path}"


# =============================================================================
# BACKGROUND SUBTRACTION
# =============================================================================

class BackgroundProcessor:
    """XRD background estimation and subtraction."""

    @staticmethod
    def rolling_minimum(intensity: np.ndarray, window: int = 151) -> np.ndarray:
        """
        Rolling minimum background estimation.

        Better than polynomial fitting because it follows the baseline
        between peaks without being pulled upward by peak intensities.

        Args:
            intensity: Raw intensity array.
            window: Window size (must be odd). Larger = smoother baseline.

        Returns:
            Background array.
        """
        if window % 2 == 0:
            window += 1
        window = max(3, min(window, len(intensity) // 3))

        # Rolling minimum, then smooth to remove staircase artefacts
        bg = minimum_filter1d(intensity, size=window, mode='nearest')

        # Smooth the rolling minimum with Savitzky-Golay
        smooth_window = min(window, len(bg))
        if smooth_window % 2 == 0:
            smooth_window -= 1
        smooth_window = max(5, smooth_window)

        bg = savgol_filter(bg, smooth_window, 3)

        # Ensure background never exceeds intensity
        bg = np.minimum(bg, intensity * 0.98)

        return bg

    @staticmethod
    def snip(intensity: np.ndarray, n_iterations: int = 40) -> np.ndarray:
        """
        SNIP (Statistics-sensitive Non-linear Iterative Peak-clipping).

        The gold standard for XRD background estimation.
        C.G. Ryan et al., Nuclear Instruments and Methods B34 (1988) 396-402.

        Args:
            intensity: Raw intensity array.
            n_iterations: Number of clipping iterations. More = smoother.

        Returns:
            Background array.
        """
        # Work in sqrt space for better statistics sensitivity
        y = np.sqrt(np.maximum(intensity, 0))
        n = len(y)

        for p in range(1, n_iterations + 1):
            y_new = y.copy()
            for i in range(p, n - p):
                y_new[i] = min(y[i], 0.5 * (y[i - p] + y[i + p]))
            y = y_new

        # Transform back from sqrt space
        bg = y ** 2
        bg = np.minimum(bg, intensity * 0.98)

        return bg

    @staticmethod
    def subtract(two_theta: np.ndarray, intensity: np.ndarray,
                 method: str = "snip", **kwargs) -> Tuple[np.ndarray, np.ndarray]:
        """
        Subtract background from XRD data.

        Args:
            two_theta: 2θ values.
            intensity: Intensity values.
            method: "snip" or "rolling_minimum".
            **kwargs: Passed to the selected method.

        Returns:
            (background_subtracted_intensity, background)
        """
        if method == "snip":
            bg = BackgroundProcessor.snip(intensity, **kwargs)
        elif method == "rolling_minimum":
            bg = BackgroundProcessor.rolling_minimum(intensity, **kwargs)
        else:
            raise ValueError(f"Unknown background method: {method}")

        subtracted = intensity - bg
        subtracted = np.maximum(subtracted, 0)

        return subtracted, bg


# =============================================================================
# PEAK FITTING
# =============================================================================

class PeakFitter:
    """
    XRD peak fitting using pseudo-Voigt profiles with optional Kα doublet.

    Uses lmfit for robust nonlinear least-squares fitting.
    """

    def __init__(self, wavelength_info: Dict):
        """
        Args:
            wavelength_info: dict with keys 'ka1', 'ka2', 'ratio'.
        """
        if not LMFIT_AVAILABLE:
            raise ImportError("lmfit is required for peak fitting.")

        self.ka1 = wavelength_info['ka1']
        self.ka2 = wavelength_info['ka2']
        self.ka_ratio = wavelength_info['ratio']

    def _delta_2theta_doublet(self, two_theta_deg: float) -> float:
        """
        Calculate theoretical Kα1-Kα2 splitting at a given 2θ.

        Uses differentiation of Bragg's law:
            Δ(2θ) = 2 * tan(θ) * (λ₂ - λ₁) / λ₁

        Returns splitting in degrees.
        """
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        if np.isclose(theta_rad, 0) or np.isclose(np.cos(theta_rad), 0):
            return 0.0

        delta_lambda = self.ka2 - self.ka1
        delta_theta_rad = np.tan(theta_rad) * (delta_lambda / self.ka1)
        return 2 * np.rad2deg(delta_theta_rad)

    @staticmethod
    def _pseudo_voigt_fwhm(sigma: float, fraction: float) -> float:
        """
        Calculate FWHM of a pseudo-Voigt from lmfit parameters.

        For lmfit PseudoVoigtModel, both Gaussian and Lorentzian
        components share the same sigma parameter:
            FWHM_G = 2σ √(2 ln 2) ≈ 2.3548 σ
            FWHM_L = 2σ

        The composite FWHM is approximated by linear interpolation:
            FWHM ≈ (1 - η) FWHM_G + η FWHM_L

        where η is the Lorentzian fraction.
        """
        fwhm_g = 2.0 * sigma * np.sqrt(2.0 * np.log(2.0))
        fwhm_l = 2.0 * sigma
        return (1.0 - fraction) * fwhm_g + fraction * fwhm_l

    @staticmethod
    def _numerical_fwhm(x: np.ndarray, y: np.ndarray) -> float:
        """
        Numerically determine FWHM from a densely evaluated peak profile.

        More robust than analytical approximation for composite profiles.
        """
        max_idx = np.argmax(y)
        half_max = y[max_idx] / 2.0

        # Find left half-max crossing
        left_idx = max_idx
        while left_idx > 0 and y[left_idx] > half_max:
            left_idx -= 1

        # Find right half-max crossing
        right_idx = max_idx
        while right_idx < len(y) - 1 and y[right_idx] > half_max:
            right_idx += 1

        # Linear interpolation for sub-point precision
        if left_idx < max_idx and y[left_idx + 1] != y[left_idx]:
            x_left = x[left_idx] + (half_max - y[left_idx]) / (y[left_idx + 1] - y[left_idx]) * (x[left_idx + 1] - x[left_idx])
        else:
            x_left = x[left_idx]

        if right_idx > max_idx and y[right_idx] != y[right_idx - 1]:
            x_right = x[right_idx - 1] + (half_max - y[right_idx - 1]) / (y[right_idx] - y[right_idx - 1]) * (x[right_idx] - x[right_idx - 1])
        else:
            x_right = x[right_idx]

        return max(x_right - x_left, 1e-6)

    def fit_peak(self, two_theta: np.ndarray, intensity: np.ndarray,
                 peak_guess_deg: float,
                 fit_window_deg: float = DEFAULT_FIT_WINDOW_DEG,
                 enable_doublet: bool = True) -> PeakFitResult:
        """
        Fit a single XRD peak with a pseudo-Voigt profile.

        If enable_doublet is True and the Kα splitting is resolvable
        (> 10% of estimated peak width), fits a constrained doublet
        with Kα2 centre, amplitude, and sigma linked to Kα1.

        Args:
            two_theta: Full 2θ array.
            intensity: Background-subtracted intensity.
            peak_guess_deg: Initial 2θ guess for peak centre.
            fit_window_deg: Total width of fitting window.
            enable_doublet: Whether to attempt Kα1-Kα2 doublet fit.

        Returns:
            PeakFitResult dataclass.
        """
        # Extract fitting window
        half_window = fit_window_deg / 2.0
        mask = (two_theta >= peak_guess_deg - half_window) & \
               (two_theta <= peak_guess_deg + half_window)
        x = two_theta[mask]
        y = intensity[mask]

        if len(x) < MIN_FIT_POINTS:
            return PeakFitResult(
                success=False,
                message=f"Insufficient data points ({len(x)}) in fitting window."
            )

        # Initial guesses
        amp_guess = np.max(y)
        sigma_guess = max(0.02, (x[-1] - x[0]) / 10.0)

        # Decide whether doublet is worth fitting
        delta_2theta = self._delta_2theta_doublet(peak_guess_deg) if enable_doublet else 0.0
        use_doublet = enable_doublet and (delta_2theta > 0.1 * sigma_guess * 2.3548)

        try:
            if use_doublet:
                result = self._fit_doublet(x, y, peak_guess_deg, amp_guess,
                                           sigma_guess, delta_2theta)
            else:
                result = self._fit_singlet(x, y, peak_guess_deg, amp_guess,
                                           sigma_guess)
        except Exception as e:
            return PeakFitResult(success=False, message=f"Fitting error: {e}")

        return result

    def _fit_singlet(self, x: np.ndarray, y: np.ndarray,
                     center_guess: float, amp_guess: float,
                     sigma_guess: float) -> PeakFitResult:
        """Fit a single pseudo-Voigt peak."""
        model = PseudoVoigtModel()
        params = model.make_params(
            amplitude=amp_guess,
            center=center_guess,
            sigma=sigma_guess,
            fraction=0.5
        )

        # Bounds
        params['center'].min = x[0]
        params['center'].max = x[-1]
        params['sigma'].min = 0.001
        params['sigma'].max = (x[-1] - x[0]) / 2.0
        params['amplitude'].min = 0
        params['fraction'].min = 0.0
        params['fraction'].max = 1.0

        fit = model.fit(y, params, x=x)

        if not fit.success:
            return PeakFitResult(success=False, message=fit.message or "Fit did not converge.")

        sigma = fit.best_values['sigma']
        fraction = fit.best_values['fraction']
        fwhm = self._pseudo_voigt_fwhm(sigma, fraction)

        # Also get numerical FWHM for verification
        fit_x_dense = np.linspace(x[0], x[-1], 500)
        fit_y_dense = fit.eval(x=fit_x_dense)

        # R² calculation
        ss_res = np.sum(fit.residual ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r_sq = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

        return PeakFitResult(
            success=True,
            center_deg=fit.best_values['center'],
            fwhm_deg=fwhm,
            amplitude=fit.best_values['amplitude'],
            fraction=fraction,
            sigma=sigma,
            r_squared=r_sq,
            reduced_chi_sq=fit.redchi if hasattr(fit, 'redchi') else 0.0,
            fit_x=fit_x_dense,
            fit_y=fit_y_dense,
            residuals=fit.residual,
            data_x=x.copy(),
            data_y=y.copy(),
            doublet_fitted=False,
            message="Single pseudo-Voigt fit."
        )

    def _fit_doublet(self, x: np.ndarray, y: np.ndarray,
                     center_guess: float, amp_guess: float,
                     sigma_guess: float, delta_2theta: float) -> PeakFitResult:
        """Fit a Kα1-Kα2 doublet with constrained pseudo-Voigt profiles."""
        model1 = PseudoVoigtModel(prefix='k1_')
        model2 = PseudoVoigtModel(prefix='k2_')
        composite = model1 + model2

        params = model1.make_params(
            amplitude=amp_guess,
            center=center_guess,
            sigma=sigma_guess,
            fraction=0.5
        )

        # Kα2 constrained to Kα1
        params.add('k2_center', expr=f'k1_center + {delta_2theta}')
        params.add('k2_amplitude', expr=f'k1_amplitude * {self.ka_ratio}')
        params.add('k2_sigma', expr='k1_sigma')
        params.add('k2_fraction', expr='k1_fraction')

        # Bounds for Kα1
        params['k1_center'].min = x[0]
        params['k1_center'].max = x[-1]
        params['k1_sigma'].min = 0.001
        params['k1_sigma'].max = (x[-1] - x[0]) / 2.0
        params['k1_amplitude'].min = 0
        params['k1_fraction'].min = 0.0
        params['k1_fraction'].max = 1.0

        fit = composite.fit(y, params, x=x)

        if not fit.success:
            # Fall back to singlet
            logger.info("Doublet fit failed, falling back to singlet.")
            return self._fit_singlet(x, y, center_guess, amp_guess, sigma_guess)

        sigma = fit.best_values['k1_sigma']
        fraction = fit.best_values['k1_fraction']
        fwhm = self._pseudo_voigt_fwhm(sigma, fraction)

        # Evaluate Kα1 component only for numerical FWHM verification
        fit_x_dense = np.linspace(x[0], x[-1], 500)
        fit_y_dense = fit.eval(x=fit_x_dense)

        # For Kα1-only FWHM, evaluate just model1
        ka1_y = model1.eval(x=fit_x_dense, **{
            'k1_amplitude': fit.best_values['k1_amplitude'],
            'k1_center': fit.best_values['k1_center'],
            'k1_sigma': sigma,
            'k1_fraction': fraction
        })
        fwhm_numerical = self._numerical_fwhm(fit_x_dense, ka1_y)

        # Use numerical FWHM if significantly different from analytical
        if abs(fwhm_numerical - fwhm) / max(fwhm, 1e-6) > 0.1:
            logger.debug(f"FWHM mismatch: analytical={fwhm:.4f}, numerical={fwhm_numerical:.4f}")

        # R²
        ss_res = np.sum(fit.residual ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r_sq = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

        return PeakFitResult(
            success=True,
            center_deg=fit.best_values['k1_center'],
            fwhm_deg=fwhm,
            amplitude=fit.best_values['k1_amplitude'],
            fraction=fraction,
            sigma=sigma,
            r_squared=r_sq,
            reduced_chi_sq=fit.redchi if hasattr(fit, 'redchi') else 0.0,
            fit_x=fit_x_dense,
            fit_y=fit_y_dense,
            residuals=fit.residual,
            data_x=x.copy(),
            data_y=y.copy(),
            doublet_fitted=True,
            message=f"Kα1-Kα2 doublet fit (Δ2θ = {delta_2theta:.4f}°)."
        )


# =============================================================================
# INSTRUMENTAL CORRECTION
# =============================================================================

class InstrumentalCorrection:
    """
    Instrumental broadening from a LaB₆ (or similar) line profile standard.

    Processes the standard pattern to extract FWHM at each standard peak,
    then provides interpolated instrumental FWHM at any 2θ position.
    """

    def __init__(self):
        self.standard_peaks: List[Dict] = []  # List of {two_theta, fwhm, r_squared}
        self._two_theta_ref: Optional[np.ndarray] = None
        self._fwhm_ref: Optional[np.ndarray] = None
        self.is_loaded = False

        # Caglioti parameters: FWHM² = U tan²θ + V tan θ + W
        self.caglioti_U: float = 0.0
        self.caglioti_V: float = 0.0
        self.caglioti_W: float = 0.0
        self.caglioti_r_squared: float = 0.0
        self.caglioti_fitted: bool = False

    def process_standard(self, two_theta: np.ndarray, intensity: np.ndarray,
                         fitter: PeakFitter,
                         bg_method: str = "snip",
                         prominence_factor: float = 0.05,
                         min_width_deg: float = 0.03,
                         fit_window_deg: float = DEFAULT_FIT_WINDOW_DEG,
                         enable_doublet: bool = True,
                         min_r_squared: float = 0.8,
                         dedup_deg: float = DEFAULT_STD_DEDUP_DEG) -> str:
        """
        Process a standard pattern to build instrumental FWHM lookup.

        Args:
            two_theta: Standard 2θ array.
            intensity: Standard intensity array.
            fitter: PeakFitter instance (for profile fitting).
            bg_method: Background subtraction method.
            prominence_factor: For peak detection.
            min_width_deg: Minimum peak width for detection.
            fit_window_deg: Fitting window width.
            enable_doublet: Whether to fit Kα doublets.
            min_r_squared: Minimum R² to accept a standard peak fit.

        Returns:
            Summary message string.
        """
        # Background subtraction
        subtracted, bg = BackgroundProcessor.subtract(
            two_theta, intensity, method=bg_method
        )

        # Peak detection
        step_size = np.median(np.diff(two_theta))
        min_width_pts = max(2, int(min_width_deg / step_size))
        min_distance = max(5, int(0.3 / step_size))

        peaks_idx, props = find_peaks(
            subtracted,
            height=np.max(subtracted) * prominence_factor,
            distance=min_distance,
            width=min_width_pts
        )

        if len(peaks_idx) == 0:
            return "No peaks detected in standard. Check parameters."

        # Fit each detected peak
        self.standard_peaks = []
        messages = []

        for idx in peaks_idx:
            guess_2theta = two_theta[idx]
            result = fitter.fit_peak(
                two_theta, subtracted, guess_2theta,
                fit_window_deg=fit_window_deg,
                enable_doublet=enable_doublet
            )

            min_r_sq = min_r_squared
            if result.success and result.r_squared > min_r_sq:
                self.standard_peaks.append({
                    'two_theta': result.center_deg,
                    'fwhm': result.fwhm_deg,
                    'r_squared': result.r_squared,
                    'fit_result': result
                })
                messages.append(
                    f"  {result.center_deg:7.3f}°: FWHM = {result.fwhm_deg:.4f}° "
                    f"(R² = {result.r_squared:.4f})"
                )
            else:
                msg = result.message if not result.success else f"R² = {result.r_squared:.3f} < {min_r_sq}"
                messages.append(f"  {guess_2theta:7.3f}°: REJECTED ({msg})")

        if self.standard_peaks:
            # Remove duplicate detections (same physical peak fitted twice)
            self.standard_peaks.sort(key=lambda p: p['two_theta'])
            deduped = [self.standard_peaks[0]]
            for p in self.standard_peaks[1:]:
                if abs(p['two_theta'] - deduped[-1]['two_theta']) < dedup_deg:
                    # Duplicate — keep the one with higher R²
                    if p['r_squared'] > deduped[-1]['r_squared']:
                        messages.append(
                            f"  {deduped[-1]['two_theta']:7.3f}°: DUPLICATE removed "
                            f"(kept {p['two_theta']:.3f}°, R² = {p['r_squared']:.4f})")
                        deduped[-1] = p
                    else:
                        messages.append(
                            f"  {p['two_theta']:7.3f}°: DUPLICATE removed "
                            f"(kept {deduped[-1]['two_theta']:.3f}°, R² = {deduped[-1]['r_squared']:.4f})")
                else:
                    deduped.append(p)
            self.standard_peaks = deduped

            # Build interpolation arrays
            self._two_theta_ref = np.array([p['two_theta'] for p in self.standard_peaks])
            self._fwhm_ref = np.array([p['fwhm'] for p in self.standard_peaks])

            # Sort by 2θ
            sort_idx = np.argsort(self._two_theta_ref)
            self._two_theta_ref = self._two_theta_ref[sort_idx]
            self._fwhm_ref = self._fwhm_ref[sort_idx]

            self.is_loaded = True

            # Fit Caglioti function: FWHM² = U tan²θ + V tan θ + W
            self._fit_caglioti(messages)

        header = f"Standard processing: {len(self.standard_peaks)}/{len(peaks_idx)} peaks accepted\n"
        return header + "\n".join(messages)

    def _fit_caglioti(self, messages: List[str]):
        """
        Fit the Caglioti function to instrumental FWHM data.

        FWHM² = U tan²θ + V tan θ + W

        where U captures high-angle broadening (strain-like / Kα splitting),
        V captures the mid-angle minimum (focusing geometry),
        W is the angle-independent contribution (slit widths, source size).

        Ref: Caglioti, Paoletti & Ricci (1958) Nucl. Instrum. 3, 223-228.
        """
        if len(self.standard_peaks) < 4:
            messages.append("  Caglioti fit: skipped (need ≥ 4 peaks)")
            return

        two_theta = np.array([p['two_theta'] for p in self.standard_peaks])
        fwhm = np.array([p['fwhm'] for p in self.standard_peaks])

        theta_rad = np.deg2rad(two_theta / 2.0)
        tan_theta = np.tan(theta_rad)

        # Dependent variable: FWHM² (in degrees²)
        fwhm_sq = fwhm ** 2

        # Design matrix: [tan²θ, tan θ, 1]
        A = np.column_stack([tan_theta ** 2, tan_theta, np.ones_like(tan_theta)])

        try:
            # Least-squares fit
            result, residuals, rank, sv = np.linalg.lstsq(A, fwhm_sq, rcond=None)
            U, V, W = result

            # R² for the Caglioti fit
            fwhm_sq_pred = U * tan_theta ** 2 + V * tan_theta + W
            ss_res = np.sum((fwhm_sq - fwhm_sq_pred) ** 2)
            ss_tot = np.sum((fwhm_sq - np.mean(fwhm_sq)) ** 2)
            r_sq = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

            # Check physical validity: W should be positive (it's FWHM² at θ=0)
            if W < 0:
                messages.append(
                    f"  Caglioti fit: W = {W:.6f} (negative) — fit is unphysical. "
                    f"Using linear interpolation instead.")
                return

            self.caglioti_U = U
            self.caglioti_V = V
            self.caglioti_W = W
            self.caglioti_r_squared = r_sq
            self.caglioti_fitted = True

            messages.append(
                f"  Caglioti fit: U = {U:.6f}, V = {V:.6f}, W = {W:.6f} "
                f"(R² = {r_sq:.4f})")

            # Report any peaks with large residuals (potential outliers)
            fwhm_pred = np.sqrt(np.maximum(fwhm_sq_pred, 0))
            for i, (pos, obs, pred) in enumerate(zip(two_theta, fwhm, fwhm_pred)):
                pct_dev = abs(obs - pred) / obs * 100
                if pct_dev > 15:
                    messages.append(
                        f"    ⚠ {pos:.1f}°: FWHM = {obs:.4f}° vs Caglioti = {pred:.4f}° "
                        f"({pct_dev:.0f}% deviation)")

        except Exception as e:
            messages.append(f"  Caglioti fit failed: {e}")

    def get_fwhm_at(self, two_theta_deg: float) -> float:
        """
        Get instrumental FWHM at a given 2θ.

        Uses Caglioti function if fitted (continuous, physically motivated),
        otherwise falls back to linear interpolation between standard peaks.

        Returns:
            Instrumental FWHM in degrees.
        """
        if not self.is_loaded or self._two_theta_ref is None:
            logger.warning("No standard data loaded. Returning 0.0 instrumental FWHM.")
            return 0.0

        if self.caglioti_fitted:
            theta_rad = np.deg2rad(two_theta_deg / 2.0)
            tan_theta = np.tan(theta_rad)
            fwhm_sq = (self.caglioti_U * tan_theta ** 2 +
                        self.caglioti_V * tan_theta +
                        self.caglioti_W)
            if fwhm_sq > 0:
                return float(np.sqrt(fwhm_sq))
            else:
                # Caglioti gives negative FWHM² at this angle — fall through
                pass

        # Fallback: linear interpolation
        if two_theta_deg <= self._two_theta_ref[0]:
            return float(self._fwhm_ref[0])
        elif two_theta_deg >= self._two_theta_ref[-1]:
            return float(self._fwhm_ref[-1])
        else:
            return float(np.interp(two_theta_deg, self._two_theta_ref, self._fwhm_ref))


# =============================================================================
# PEAK PROCESSING PIPELINE
# =============================================================================

class PeakProcessor:
    """
    Complete pipeline: detect peaks in sample, fit them, apply
    instrumental correction, produce AnalysedPeak objects.
    """

    @staticmethod
    def process_sample(two_theta: np.ndarray, intensity: np.ndarray,
                       fitter: PeakFitter,
                       inst_correction: Optional[InstrumentalCorrection] = None,
                       wavelength_nm: float = 0.154056,
                       bg_method: str = "snip",
                       prominence_factor: float = 0.05,
                       min_width_deg: float = 0.05,
                       fit_window_deg: float = DEFAULT_FIT_WINDOW_DEG,
                       enable_doublet: bool = True,
                       std_peak_centers: Optional[List[float]] = None,
                       proximity_tolerance_deg: float = 5.0,
                       min_r_squared: float = DEFAULT_MIN_R_SQUARED,
                       min_snr: float = DEFAULT_MIN_SNR,
                       dedup_deg: float = DEFAULT_SAMPLE_DEDUP_DEG,
                       snip_iterations: int = DEFAULT_SNIP_ITERATIONS,
                       ) -> Tuple[List[AnalysedPeak], List[PeakFitResult], str]:
        """
        Full sample peak processing pipeline.

        1. Background subtraction
        2. Peak detection
        3. Optional filtering by proximity to standard peaks
        4. Peak fitting (pseudo-Voigt ± doublet)
        5. Instrumental broadening correction (Gaussian deconvolution)
        6. Resolution-limit flagging

        Args:
            two_theta: Sample 2θ array.
            intensity: Sample raw intensity array.
            fitter: PeakFitter instance.
            inst_correction: InstrumentalCorrection instance (or None).
            wavelength_nm: Kα1 wavelength in nm.
            bg_method: Background subtraction method.
            prominence_factor: Peak detection prominence threshold.
            min_width_deg: Minimum peak width for detection.
            fit_window_deg: Fitting window width.
            enable_doublet: Whether to fit Kα doublets.
            std_peak_centers: If provided, only sample peaks near these
                              positions (within proximity_tolerance_deg) are kept.
            proximity_tolerance_deg: Tolerance for standard peak matching.

        Returns:
            (analysed_peaks, fit_results, log_message)
        """
        messages = []

        # Background subtraction
        bg_kwargs = {'n_iterations': snip_iterations} if bg_method == 'snip' else {}
        subtracted, bg = BackgroundProcessor.subtract(
            two_theta, intensity, method=bg_method, **bg_kwargs
        )

        # Peak detection
        step_size = np.median(np.diff(two_theta))
        min_width_pts = max(2, int(min_width_deg / step_size))
        min_distance = max(5, int(0.2 / step_size))

        peaks_idx, props = find_peaks(
            subtracted,
            height=np.max(subtracted) * prominence_factor,
            distance=min_distance,
            width=min_width_pts
        )

        if len(peaks_idx) == 0:
            return [], [], "No peaks detected in sample."

        messages.append(f"Detected {len(peaks_idx)} raw peaks.")

        # Optional proximity filtering
        if std_peak_centers is not None and len(std_peak_centers) > 0:
            filtered_idx = []
            for idx in peaks_idx:
                sample_pos = two_theta[idx]
                if any(abs(sample_pos - sp) <= proximity_tolerance_deg for sp in std_peak_centers):
                    filtered_idx.append(idx)
            peaks_idx = np.array(filtered_idx)
            messages.append(f"After proximity filtering: {len(peaks_idx)} peaks.")

        if len(peaks_idx) == 0:
            return [], [], "\n".join(messages) + "\nNo peaks after filtering."

        # Fit each peak and build AnalysedPeak objects
        analysed = []
        fit_results = []

        # Pre-compute noise level for SNR screening
        noise_level = np.std(subtracted[subtracted < np.percentile(subtracted, 30)])
        if noise_level <= 0:
            noise_level = 1.0

        for idx in peaks_idx:
            guess = two_theta[idx]
            peak_height = subtracted[idx]

            # Quick SNR pre-screen — skip noise spikes before expensive fitting
            if peak_height / noise_level < min_snr:
                continue

            # Adaptive fit window: estimate FWHM from half-max crossing,
            # then use ~4× FWHM as window (floor at 0.5°, ceiling at user value)
            half_max = peak_height / 2.0
            left_i = idx
            while left_i > 0 and subtracted[left_i] > half_max:
                left_i -= 1
            right_i = idx
            while right_i < len(subtracted) - 1 and subtracted[right_i] > half_max:
                right_i += 1
            est_fwhm = two_theta[right_i] - two_theta[left_i]
            # Cap estimate at 1° — larger usually means the walk crossed a neighbour
            est_fwhm = min(est_fwhm, 1.0)
            adaptive_window = max(0.5, min(4.0 * est_fwhm, fit_window_deg))

            result = fitter.fit_peak(
                two_theta, subtracted, guess,
                fit_window_deg=adaptive_window,
                enable_doublet=enable_doublet
            )
            fit_results.append(result)

            if not result.success:
                messages.append(f"  {guess:.3f}°: fit failed ({result.message})")
                continue

            if result.r_squared < min_r_squared:
                messages.append(
                    f"  {result.center_deg:.3f}°: poor fit (R² = {result.r_squared:.3f}), skipping."
                )
                continue

            # Instrumental correction
            fwhm_obs = result.fwhm_deg
            if inst_correction is not None and inst_correction.is_loaded:
                fwhm_inst = inst_correction.get_fwhm_at(result.center_deg)
            else:
                fwhm_inst = 0.0

            # Broadening ratio and resolution-limit check
            if fwhm_inst > 0:
                broadening_ratio = fwhm_obs / fwhm_inst
            else:
                broadening_ratio = float('inf')

            resolution_limited = broadening_ratio < RESOLUTION_LIMIT_RATIO

            # Gaussian deconvolution: β² = β²_obs - β²_inst
            if fwhm_obs > fwhm_inst:
                beta_corrected = np.sqrt(fwhm_obs ** 2 - fwhm_inst ** 2)
            else:
                beta_corrected = 0.0
                resolution_limited = True

            theta_rad = np.deg2rad(result.center_deg / 2.0)
            d_spacing = wavelength_nm / (2.0 * np.sin(theta_rad)) if np.sin(theta_rad) > 0 else 0.0

            peak = AnalysedPeak(
                two_theta_deg=result.center_deg,
                theta_rad=theta_rad,
                fwhm_obs_deg=fwhm_obs,
                fwhm_inst_deg=fwhm_inst,
                beta_corrected_deg=beta_corrected,
                beta_corrected_rad=np.deg2rad(beta_corrected),
                resolution_limited=resolution_limited,
                broadening_ratio=broadening_ratio,
                fit_quality=result.r_squared,
                d_spacing_nm=d_spacing,
                intensity=result.amplitude
            )
            analysed.append(peak)

            flag = " [RESOLUTION LIMITED]" if resolution_limited else ""
            messages.append(
                f"  {result.center_deg:7.3f}°: β_obs={fwhm_obs:.4f}° "
                f"β_inst={fwhm_inst:.4f}° β_corr={beta_corrected:.4f}° "
                f"R²={result.r_squared:.4f}{flag}"
            )

        n_skipped = len(peaks_idx) - len(fit_results)
        messages.insert(1, f"Successfully fitted: {len(analysed)}/{len(fit_results)} attempted "
                        f"({n_skipped} pre-screened out by SNR).")

        # Deduplicate: merge detections within dedup_deg keeping higher R²
        if len(analysed) > 1:
            analysed.sort(key=lambda p: p.two_theta_deg)
            deduped = [analysed[0]]
            for p in analysed[1:]:
                if abs(p.two_theta_deg - deduped[-1].two_theta_deg) < dedup_deg:
                    if p.fit_quality > deduped[-1].fit_quality:
                        messages.append(
                            f"  {deduped[-1].two_theta_deg:7.3f}°: DUPLICATE removed "
                            f"(kept {p.two_theta_deg:.3f}°, R²={p.fit_quality:.4f})")
                        deduped[-1] = p
                    else:
                        messages.append(
                            f"  {p.two_theta_deg:7.3f}°: DUPLICATE removed "
                            f"(kept {deduped[-1].two_theta_deg:.3f}°, R²={deduped[-1].fit_quality:.4f})")
                else:
                    deduped.append(p)
            analysed = deduped

        return analysed, fit_results, "\n".join(messages)

    @staticmethod
    def fit_manual_peak(two_theta: np.ndarray, intensity: np.ndarray,
                        peak_guess_deg: float,
                        fitter: 'PeakFitter',
                        inst_correction: Optional['InstrumentalCorrection'] = None,
                        wavelength_nm: float = 0.154056,
                        bg_method: str = "snip",
                        fit_window_deg: float = DEFAULT_FIT_WINDOW_DEG,
                        enable_doublet: bool = True
                        ) -> Tuple[Optional['AnalysedPeak'], Optional['PeakFitResult'], str]:
        """
        Fit a single peak at a user-specified 2θ position.

        Used for click-to-add-peak in the GUI. Background-subtracts the
        data, fits a pseudo-Voigt at the specified position, applies
        instrumental correction, and returns a single AnalysedPeak.

        Args:
            two_theta: Full 2θ array.
            intensity: Raw intensity array.
            peak_guess_deg: User-specified 2θ guess.
            fitter: PeakFitter instance.
            inst_correction: InstrumentalCorrection instance (or None).
            wavelength_nm: Kα1 wavelength in nm.
            bg_method: Background subtraction method.
            fit_window_deg: Fitting window width.
            enable_doublet: Whether to fit Kα doublets.

        Returns:
            (AnalysedPeak or None, PeakFitResult or None, log_message)
        """
        subtracted, bg = BackgroundProcessor.subtract(
            two_theta, intensity, method=bg_method)

        result = fitter.fit_peak(
            two_theta, subtracted, peak_guess_deg,
            fit_window_deg=fit_window_deg,
            enable_doublet=enable_doublet)

        if not result.success:
            return None, result, f"Fit failed at {peak_guess_deg:.3f}\u00b0: {result.message}"

        # For manual peaks, always accept — user explicitly requested this position.
        # Flag poor quality but don't reject.
        quality_warning = ""
        if result.r_squared < 0.5:
            quality_warning = " [POOR FIT — inspect carefully]"

        fwhm_obs = result.fwhm_deg
        if inst_correction is not None and inst_correction.is_loaded:
            fwhm_inst = inst_correction.get_fwhm_at(result.center_deg)
        else:
            fwhm_inst = 0.0

        broadening_ratio = fwhm_obs / fwhm_inst if fwhm_inst > 0 else float('inf')
        resolution_limited = broadening_ratio < RESOLUTION_LIMIT_RATIO

        if fwhm_obs > fwhm_inst:
            beta_corrected = np.sqrt(fwhm_obs ** 2 - fwhm_inst ** 2)
        else:
            beta_corrected = 0.0
            resolution_limited = True

        theta_rad = np.deg2rad(result.center_deg / 2.0)
        d_spacing = wavelength_nm / (2.0 * np.sin(theta_rad)) if np.sin(theta_rad) > 0 else 0.0

        peak = AnalysedPeak(
            two_theta_deg=result.center_deg,
            theta_rad=theta_rad,
            fwhm_obs_deg=fwhm_obs,
            fwhm_inst_deg=fwhm_inst,
            beta_corrected_deg=beta_corrected,
            beta_corrected_rad=np.deg2rad(beta_corrected),
            resolution_limited=resolution_limited,
            broadening_ratio=broadening_ratio,
            fit_quality=result.r_squared,
            d_spacing_nm=d_spacing,
            intensity=result.amplitude
        )

        flag = " [RESOLUTION LIMITED]" if resolution_limited else ""
        msg = (f"Manual peak at {result.center_deg:.3f}\u00b0: "
               f"\u03b2_obs={fwhm_obs:.4f}\u00b0 \u03b2_inst={fwhm_inst:.4f}\u00b0 "
               f"\u03b2_corr={beta_corrected:.4f}\u00b0 R\u00b2={result.r_squared:.4f}"
               f"{flag}{quality_warning}")

        return peak, result, msg


# =============================================================================
# ANALYSIS METHOD 1: SCHERRER
# =============================================================================

class ScherrerAnalyzer:
    """
    Standard Scherrer equation: D = Kλ / (β cos θ)

    where:
        D = crystallite size (nm)
        K = shape factor (dimensionless)
        λ = wavelength (nm)
        β = corrected FWHM (radians)
        θ = Bragg angle (radians)

    Includes resolution-limit detection and per-peak reporting.
    """

    def __init__(self, wavelength_nm: float):
        self.wavelength = wavelength_nm

    def analyse(self, peaks: List[AnalysedPeak],
                k_constant: float = 0.94) -> Dict[str, Any]:
        """
        Run Scherrer analysis on all peaks.

        Returns dict with 'success', 'results', 'average_size_nm',
        'std_size_nm', 'k_constant', warnings, etc.
        """
        if not peaks:
            return {'success': False, 'method': 'Scherrer',
                    'message': 'No peaks provided.'}

        results = []

        for peak in peaks:
            if peak.resolution_limited or peak.beta_corrected_rad <= 0:
                # Report lower bound
                beta_rad = np.deg2rad(peak.fwhm_inst_deg) if peak.fwhm_inst_deg > 0 else np.deg2rad(peak.fwhm_obs_deg)
                cos_theta = np.cos(peak.theta_rad)
                if beta_rad > 0 and cos_theta > 0:
                    lower_bound = k_constant * self.wavelength / (beta_rad * cos_theta)
                else:
                    lower_bound = np.nan

                results.append({
                    'two_theta': peak.two_theta_deg,
                    'size_nm': lower_bound,
                    'fwhm_obs': peak.fwhm_obs_deg,
                    'fwhm_inst': peak.fwhm_inst_deg,
                    'beta_corrected': peak.beta_corrected_deg,
                    'resolution_limited': True,
                    'broadening_ratio': peak.broadening_ratio,
                    'fit_quality': peak.fit_quality,
                })
            else:
                cos_theta = np.cos(peak.theta_rad)
                if cos_theta > 0:
                    size = k_constant * self.wavelength / (peak.beta_corrected_rad * cos_theta)
                else:
                    size = np.nan

                results.append({
                    'two_theta': peak.two_theta_deg,
                    'size_nm': size,
                    'fwhm_obs': peak.fwhm_obs_deg,
                    'fwhm_inst': peak.fwhm_inst_deg,
                    'beta_corrected': peak.beta_corrected_deg,
                    'resolution_limited': False,
                    'broadening_ratio': peak.broadening_ratio,
                    'fit_quality': peak.fit_quality,
                })

        # Statistics
        measurable = [r['size_nm'] for r in results
                      if not r['resolution_limited'] and not np.isnan(r['size_nm'])]
        all_sizes = [r['size_nm'] for r in results if not np.isnan(r['size_nm'])]
        n_res_limited = sum(1 for r in results if r['resolution_limited'])

        output = {
            'success': True,
            'method': 'Scherrer',
            'results': results,
            'k_constant': k_constant,
            'n_peaks': len(results),
            'n_resolution_limited': n_res_limited,
        }

        if measurable:
            output['average_size_nm'] = np.mean(measurable)
            output['std_size_nm'] = np.std(measurable)
            output['n_measurable'] = len(measurable)

            # Population analysis: split by median, report group statistics
            if len(measurable) >= 4:
                median_d = np.median(measurable)
                # Match sizes to 2theta for correlation
                size_angle_pairs = [(r['size_nm'], r['two_theta'])
                                    for r in results
                                    if not r['resolution_limited'] and not np.isnan(r['size_nm'])]
                sizes_arr = np.array([s for s, _ in size_angle_pairs])
                thetas_arr = np.array([np.sin(np.deg2rad(t / 2)) for _, t in size_angle_pairs])

                above = [s for s in measurable if s > median_d]
                below = [s for s in measurable if s <= median_d]

                populations = {}
                if above:
                    populations['large'] = {
                        'mean': np.mean(above), 'std': np.std(above),
                        'n': len(above), 'range': (min(above), max(above)),
                    }
                if below:
                    populations['small'] = {
                        'mean': np.mean(below), 'std': np.std(below),
                        'n': len(below), 'range': (min(below), max(below)),
                    }
                populations['median'] = median_d

                # Angle correlation: negative r → strain contribution
                if len(sizes_arr) >= 3:
                    r_corr = float(np.corrcoef(thetas_arr, sizes_arr)[0, 1])
                    populations['angle_correlation'] = r_corr

                output['populations'] = populations
        elif all_sizes:
            output['average_size_nm'] = np.mean(all_sizes)
            output['std_size_nm'] = np.std(all_sizes)
            output['n_measurable'] = 0
            output['warning'] = (
                f"All {len(results)} peaks are resolution-limited. "
                f"Reported sizes are lower bounds."
            )
        else:
            output['success'] = False
            output['message'] = "No valid crystallite sizes could be calculated."

        if n_res_limited > 0 and measurable:
            output['warning'] = (
                f"{n_res_limited}/{len(results)} peaks are resolution-limited "
                f"and excluded from the average."
            )

        return output


# =============================================================================
# ANALYSIS METHOD 2: MONSHI-SCHERRER
# =============================================================================

class MonshiScherrerAnalyzer:
    """
    Monshi-Scherrer modified Scherrer method (Monshi et al. 2012).

    Starting from the Scherrer equation:
        β = Kλ / (L cos θ)

    Taking natural logarithms:
        ln(β) = ln(Kλ/L) + ln(1/cos θ)

    A plot of ln(β) vs ln(1/cos θ) should be linear with:
        - slope ≈ 1.0 (quality check: deviation indicates systematic error)
        - intercept = ln(Kλ/L), so L = Kλ / exp(intercept)

    This method uses data from ALL peaks simultaneously via linear
    regression, reducing the influence of random errors in individual
    peak widths.

    Reference:
        Monshi, A., Foroughi, M.R. & Monshi, M. (2012).
        World J. Nano Sci. Eng. 2, 154-160.
    """

    def __init__(self, wavelength_nm: float):
        self.wavelength = wavelength_nm

    def analyse(self, peaks: List[AnalysedPeak],
                k_constant: float = 0.94) -> Dict[str, Any]:
        """
        Run Monshi-Scherrer analysis.

        Returns dict including crystallite_size_nm, slope (should ≈ 1),
        r_squared, and the linear fit data.
        """
        # Use only non-resolution-limited peaks with valid corrected broadening
        valid_peaks = [p for p in peaks
                       if not p.resolution_limited and p.beta_corrected_rad > 0]

        if len(valid_peaks) < 2:
            return {'success': False, 'method': 'Monshi-Scherrer',
                    'message': f'Need ≥ 2 valid peaks, have {len(valid_peaks)}.'}

        # Build arrays: ln(β) vs ln(1/cos θ)
        x_data = []  # ln(1/cos θ)
        y_data = []  # ln(β) where β is in radians

        for peak in valid_peaks:
            cos_theta = np.cos(peak.theta_rad)
            if cos_theta > 0 and peak.beta_corrected_rad > 0:
                x_data.append(np.log(1.0 / cos_theta))
                y_data.append(np.log(peak.beta_corrected_rad))

        if len(x_data) < 2:
            return {'success': False, 'method': 'Monshi-Scherrer',
                    'message': 'Insufficient valid data after filtering.'}

        x = np.array(x_data)
        y = np.array(y_data)

        # Linear regression
        coeffs = np.polyfit(x, y, 1)
        slope, intercept = coeffs

        # R²
        y_pred = slope * x + intercept
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

        # Extract crystallite size from intercept
        # intercept = ln(Kλ/L) → L = Kλ / exp(intercept)
        crystallite_size_nm = k_constant * self.wavelength / np.exp(intercept)

        # Quality check: slope should be ≈ 1.0
        slope_deviation = abs(slope - 1.0)
        slope_quality = "good" if slope_deviation < 0.2 else \
                        "acceptable" if slope_deviation < 0.5 else "poor"

        output = {
            'success': True,
            'method': 'Monshi-Scherrer',
            'crystallite_size_nm': crystallite_size_nm,
            'slope': slope,
            'intercept': intercept,
            'r_squared': r_squared,
            'k_constant': k_constant,
            'n_peaks': len(x),
            'slope_quality': slope_quality,
            'linear_fit': LinearFitResult(
                slope=slope, intercept=intercept, r_squared=r_squared,
                x_data=x, y_data=y, n_points=len(x)
            ),
        }

        if slope_quality == "poor":
            output['warning'] = (
                f"Slope = {slope:.3f} deviates significantly from 1.0. "
                f"This indicates systematic errors in the FWHM values or "
                f"significant anisotropic broadening."
            )

        return output


# =============================================================================
# ANALYSIS METHOD 3: WILLIAMSON-HALL (UDM)
# =============================================================================

class WilliamsonHallAnalyzer:
    """
    Williamson-Hall Uniform Deformation Model (UDM).

    The total broadening has contributions from size and strain:
        β cos θ = Kλ/D + 4ε sin θ

    A plot of β cos θ (y-axis) vs 4 sin θ (x-axis) gives:
        - intercept = Kλ/D → D = Kλ / intercept
        - slope = ε (microstrain)

    Note: β and angles are in radians for the calculation.
    The axes are NOT normalised by λ (that would change the intercept meaning).

    Reference:
        Williamson, G.K. & Hall, W.H. (1953). Acta Metallurgica 1, 22-31.
    """

    def __init__(self, wavelength_nm: float):
        self.wavelength = wavelength_nm

    def analyse(self, peaks: List[AnalysedPeak],
                k_constant: float = 0.94) -> Dict[str, Any]:
        """
        Run Williamson-Hall analysis.

        Requires ≥ 3 non-resolution-limited peaks.
        """
        valid_peaks = [p for p in peaks
                       if not p.resolution_limited and p.beta_corrected_rad > 0]

        if len(valid_peaks) < 3:
            return {'success': False, 'method': 'Williamson-Hall',
                    'message': f'Need ≥ 3 valid peaks, have {len(valid_peaks)}.'}

        # Build arrays
        x_data = []  # 4 sin θ
        y_data = []  # β cos θ (both in radians/dimensionless)

        for peak in valid_peaks:
            x_data.append(4.0 * np.sin(peak.theta_rad))
            y_data.append(peak.beta_corrected_rad * np.cos(peak.theta_rad))

        x = np.array(x_data)
        y = np.array(y_data)

        # Linear regression
        coeffs = np.polyfit(x, y, 1)
        slope, intercept = coeffs

        # R²
        y_pred = slope * x + intercept
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

        # Extract physical parameters
        # intercept = Kλ/D (radians * nm / nm = radians)
        if intercept > 0:
            crystallite_size_nm = k_constant * self.wavelength / intercept
        else:
            crystallite_size_nm = float('inf')

        # slope = ε (dimensionless microstrain)
        microstrain = slope
        microstrain_percent = microstrain * 100

        output = {
            'success': True,
            'method': 'Williamson-Hall',
            'crystallite_size_nm': crystallite_size_nm,
            'microstrain': microstrain,
            'microstrain_percent': microstrain_percent,
            'r_squared': r_squared,
            'k_constant': k_constant,
            'n_peaks': len(x),
            'linear_fit': LinearFitResult(
                slope=slope, intercept=intercept, r_squared=r_squared,
                x_data=x, y_data=y, n_points=len(x)
            ),
        }

        if intercept <= 0:
            output['warning'] = (
                f"Negative intercept ({intercept:.4e}): crystallite size "
                f"cannot be determined. This may indicate dominant strain "
                f"broadening or insufficient angular range."
            )

        if r_squared < 0.5:
            output.setdefault('warning', '')
            output['warning'] += (
                f" Low R² = {r_squared:.3f}: W-H plot shows poor linearity. "
                f"This may indicate anisotropic broadening or mixed "
                f"broadening mechanisms."
            )

        return output


# =============================================================================
# ANALYSIS METHOD 4: HALDER-WAGNER
# =============================================================================

class HalderWagnerAnalyzer:
    """
    Halder-Wagner method for Voigt-profiled broadening.

    Assumes the peak profile is a Voigt function (convolution of
    Gaussian strain broadening and Lorentzian size broadening).

    The H-W approximation for the total integral breadth:
        β² ≈ β_L · β + β_G²

    In reciprocal units (β* = β cos θ/λ, d* = 2 sin θ/λ):
        (β*/d*)² = (K/D) · (β*/(d*)²) + (2ε)²

    Plot: y = (β*/d*)² vs x = β*/(d*)²
        - slope = K/D → D = K/slope (nm)
        - intercept = (2ε)² → ε = √(intercept) / 2

    This method gives more weight to low- and medium-angle peaks,
    reducing the influence of high-angle peaks where errors are larger.

    Reference:
        Halder, N.C. & Wagner, C.N.J. (1966). Acta Cryst. 20, 312-313.
    """

    def __init__(self, wavelength_nm: float):
        self.wavelength = wavelength_nm

    def analyse(self, peaks: List[AnalysedPeak],
                k_constant: float = 0.94) -> Dict[str, Any]:
        """
        Run Halder-Wagner analysis.

        Requires ≥ 3 non-resolution-limited peaks.
        """
        valid_peaks = [p for p in peaks
                       if not p.resolution_limited and p.beta_corrected_rad > 0]

        if len(valid_peaks) < 3:
            return {'success': False, 'method': 'Halder-Wagner',
                    'message': f'Need ≥ 3 valid peaks, have {len(valid_peaks)}.'}

        x_data = []  # β* / (d*)²
        y_data = []  # (β* / d*)²

        for peak in valid_peaks:
            beta_star = peak.beta_corrected_rad * np.cos(peak.theta_rad) / self.wavelength
            d_star = 2.0 * np.sin(peak.theta_rad) / self.wavelength

            if d_star > 0 and beta_star > 0:
                x_data.append(beta_star / (d_star ** 2))
                y_data.append((beta_star / d_star) ** 2)

        if len(x_data) < 3:
            return {'success': False, 'method': 'Halder-Wagner',
                    'message': 'Insufficient valid data after filtering.'}

        x = np.array(x_data)
        y = np.array(y_data)

        # Linear regression
        coeffs = np.polyfit(x, y, 1)
        slope, intercept = coeffs

        # R²
        y_pred = slope * x + intercept
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

        # Extract physical parameters
        # slope = K/D in nm⁻¹ → D = K/slope in nm
        if slope > 0:
            crystallite_size_nm = k_constant / slope
        else:
            crystallite_size_nm = float('inf')

        # intercept = (2ε)² → ε = √(intercept) / 2
        if intercept > 0:
            microstrain = np.sqrt(intercept) / 2.0
        elif intercept < 0:
            # Negative intercept: physically means negligible strain
            microstrain = 0.0
        else:
            microstrain = 0.0

        microstrain_percent = microstrain * 100

        output = {
            'success': True,
            'method': 'Halder-Wagner',
            'crystallite_size_nm': crystallite_size_nm,
            'microstrain': microstrain,
            'microstrain_percent': microstrain_percent,
            'r_squared': r_squared,
            'k_constant': k_constant,
            'n_peaks': len(x),
            'linear_fit': LinearFitResult(
                slope=slope, intercept=intercept, r_squared=r_squared,
                x_data=x, y_data=y, n_points=len(x)
            ),
        }

        if slope <= 0:
            output['warning'] = (
                f"Non-positive slope ({slope:.4e}): crystallite size "
                f"cannot be determined from Halder-Wagner plot."
            )

        if intercept < 0:
            output.setdefault('warning', '')
            output['warning'] += (
                f" Negative intercept ({intercept:.4e}) implies negligible "
                f"or zero strain. Strain set to 0."
            )

        return output


# =============================================================================
# ANALYSIS METHOD 5: SIZE-STRAIN PLOT (SSP)
# =============================================================================

class SSPAnalyzer:
    """
    Size-Strain Plot (SSP) method.

    The SSP is analytically equivalent to the Halder-Wagner method
    when formulated correctly (see Khorsand Zak et al., J. Alloys Compd.
    2025 for discussion of the dimensional consistency issue).

    Traditional SSP formulation (widely used in literature):
        (d²_hkl · β cos θ)² = (Kλ/D) · (d²_hkl · β cos θ) + (ε/2)²

    NOTE: Some published SSP formulations have dimensional inconsistencies.
    We implement the standard form used in literature for comparability,
    with a note on the H-W equivalence.

    Plot: y = (d_hkl β cos θ)² vs x = d²_hkl β cos θ
        - slope = Kλ/D → D = Kλ/slope
        - intercept = (ε/2)²  → ε = 2√(intercept)

    Here d_hkl is in nm, β is in radians.
    """

    def __init__(self, wavelength_nm: float):
        self.wavelength = wavelength_nm

    def analyse(self, peaks: List[AnalysedPeak],
                k_constant: float = 0.94) -> Dict[str, Any]:
        """
        Run Size-Strain Plot analysis.

        Requires ≥ 3 non-resolution-limited peaks.
        """
        valid_peaks = [p for p in peaks
                       if not p.resolution_limited and p.beta_corrected_rad > 0]

        if len(valid_peaks) < 3:
            return {'success': False, 'method': 'SSP',
                    'message': f'Need ≥ 3 valid peaks, have {len(valid_peaks)}.'}

        x_data = []  # d²_hkl · β cos θ
        y_data = []  # (d_hkl · β cos θ)²

        for peak in valid_peaks:
            d = peak.d_spacing_nm
            beta = peak.beta_corrected_rad
            cos_theta = np.cos(peak.theta_rad)

            if d > 0 and beta > 0:
                beta_cos = beta * cos_theta
                x_data.append(d ** 2 * beta_cos)
                y_data.append((d * beta_cos) ** 2)

        if len(x_data) < 3:
            return {'success': False, 'method': 'SSP',
                    'message': 'Insufficient valid data after filtering.'}

        x = np.array(x_data)
        y = np.array(y_data)

        # Linear regression
        coeffs = np.polyfit(x, y, 1)
        slope, intercept = coeffs

        # R²
        y_pred = slope * x + intercept
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

        # Extract physical parameters
        # slope = Kλ/D → D = Kλ/slope
        if slope > 0:
            crystallite_size_nm = k_constant * self.wavelength / slope
        else:
            crystallite_size_nm = float('inf')

        # intercept = (ε/2)² → ε = 2√(intercept)
        if intercept > 0:
            microstrain = 2.0 * np.sqrt(intercept)
        else:
            microstrain = 0.0

        microstrain_percent = microstrain * 100

        output = {
            'success': True,
            'method': 'SSP',
            'crystallite_size_nm': crystallite_size_nm,
            'microstrain': microstrain,
            'microstrain_percent': microstrain_percent,
            'r_squared': r_squared,
            'k_constant': k_constant,
            'n_peaks': len(x),
            'note': (
                'The SSP method is analytically equivalent to the '
                'Halder-Wagner method (same underlying Voigt approximation). '
                'Results should be consistent between the two.'
            ),
            'linear_fit': LinearFitResult(
                slope=slope, intercept=intercept, r_squared=r_squared,
                x_data=x, y_data=y, n_points=len(x)
            ),
        }

        if slope <= 0:
            output['warning'] = (
                f"Non-positive slope ({slope:.4e}): crystallite size "
                f"cannot be determined from SSP plot."
            )

        return output


# =============================================================================
# RESULTS FORMATTER
# =============================================================================

class ResultsFormatter:
    """Format analysis results with pedagogical context."""

    # Method notes — assumptions, limitations, and guidance
    NOTES = {
        'Scherrer': (
            "ASSUMPTIONS: Broadening is due to finite crystallite size only. "
            "No contribution from microstrain, stacking faults, or other "
            "defects. All crystallites are the same size (no distribution). "
            "The Scherrer equation gives a volume-averaged column length, "
            "NOT a particle size.\n"
            "LIMITATIONS: Only valid below ~100-200 nm (above this, peak "
            "broadening approaches instrumental resolution). Assumes "
            "isotropic broadening — if D varies significantly with (hkl), "
            "the average is physically meaningless. The K constant depends "
            "on crystallite shape and the breadth measure used. "
            "Cannot separate size from strain broadening.\n"
            "QUALITY CHECK: If the standard deviation exceeds ~20% of the "
            "mean, the isotropic assumption is violated. Check the D vs 2θ "
            "plot for systematic trends."
        ),
        'Monshi-Scherrer': (
            "ASSUMPTIONS: Same as Scherrer, plus the assumption that "
            "ln(β) varies linearly with ln(1/cos θ) with slope = 1. "
            "Uses all peaks simultaneously via linear regression, reducing "
            "the influence of random errors in individual FWHM values.\n"
            "LIMITATIONS: The slope = 1 constraint is a built-in quality "
            "check. Slope >> 1 indicates systematic FWHM errors or "
            "anisotropic broadening. Cannot separate size from strain.\n"
            "QUALITY CHECK: Slope should be 1.0 ± 0.2 for reliable results. "
            "If slope > 1.5, the Scherrer model is inadequate for this "
            "material.\n"
            "REF: Monshi, Foroughi & Monshi (2012) WJNSE 2, 154-160."
        ),
        'Williamson-Hall': (
            "ASSUMPTIONS: Total broadening is a LINEAR SUM of size and "
            "strain contributions (Cauchy/Lorentzian profiles). All strain "
            "is isotropic (uniform deformation model, UDM). This is the "
            "simplest size-strain separation method.\n"
            "LIMITATIONS: The linear sum assumption is only valid for "
            "purely Lorentzian profiles. Real XRD peaks are Voigt "
            "(Gaussian + Lorentzian), so the W-H method systematically "
            "underestimates size and overestimates strain. Highly sensitive "
            "to scatter — poor linearity (low R²) is common, especially "
            "with anisotropic materials. A negative intercept means size "
            "cannot be determined.\n"
            "QUALITY CHECK: R² > 0.9 suggests isotropic broadening. "
            "R² < 0.5 indicates the UDM assumption is violated — consider "
            "Halder-Wagner or modified W-H models (USDM, UDEDM).\n"
            "REF: Williamson & Hall (1953) Acta Metallurgica 1, 22-31."
        ),
        'Halder-Wagner': (
            "ASSUMPTIONS: Peak profiles are Voigt functions (convolution "
            "of Gaussian strain broadening and Lorentzian size broadening). "
            "This is physically more realistic than W-H.\n"
            "LIMITATIONS: Gives more weight to low- and mid-angle peaks "
            "where data quality is usually better. Less sensitive to "
            "high-angle noise than W-H. However, still assumes isotropic "
            "broadening. A negative intercept implies negligible strain.\n"
            "QUALITY CHECK: R² should be higher than the corresponding "
            "W-H value. If H-W R² is also poor, the material has genuine "
            "anisotropic broadening that no isotropic method can capture.\n"
            "REF: Halder & Wagner (1966) Acta Cryst. 20, 312-313."
        ),
        'SSP': (
            "ASSUMPTIONS: Identical to Halder-Wagner — Voigt profile "
            "approximation with Lorentzian size and Gaussian strain "
            "broadening.\n"
            "LIMITATIONS: The SSP and H-W methods are analytically "
            "equivalent (same underlying equation, different axis "
            "transformation). Results should be identical for D and "
            "very similar for strain. Any difference in strain values "
            "is due to different weighting of data points in the "
            "regression, not different physics.\n"
            "NOTE: Some published SSP formulations have dimensional "
            "inconsistencies. This implementation uses the correct "
            "Halder-Wagner-derived form.\n"
            "REF: See Khorsand Zak et al. (2025) J. Alloys Compd. for "
            "discussion of H-W/SSP equivalence."
        ),
    }

    @staticmethod
    def format_scherrer(result: Dict) -> str:
        if not result.get('success'):
            return f"Scherrer Analysis: {result.get('message', 'Failed')}"

        lines = [
            "=" * 60,
            "SCHERRER ANALYSIS",
            "=" * 60,
            f"Shape factor K = {result['k_constant']}",
            f"Peaks analysed: {result['n_peaks']}",
            f"Resolution-limited: {result.get('n_resolution_limited', 0)}",
            "-" * 60,
        ]

        for r in result['results']:
            flag = " ** RESOLUTION LIMITED **" if r['resolution_limited'] else ""
            size_str = f"{r['size_nm']:.1f}" if not np.isnan(r['size_nm']) else "N/A"
            lines.append(
                f"  2θ = {r['two_theta']:7.3f}°  |  "
                f"β_obs = {r['fwhm_obs']:.4f}°  |  "
                f"β_inst = {r['fwhm_inst']:.4f}°  |  "
                f"β_corr = {r['beta_corrected']:.4f}°  |  "
                f"D = {size_str} nm{flag}"
            )

        lines.append("-" * 60)
        if 'average_size_nm' in result:
            avg = result['average_size_nm']
            std = result.get('std_size_nm', 0)
            cv = (std / avg * 100) if avg > 0 else 0
            lines.append(f"Average crystallite size: {avg:.1f} \u00b1 {std:.1f} nm (CV = {cv:.0f}%)")

        # Population analysis
        pops = result.get('populations')
        if pops and cv > 20:
            lines.append("")
            lines.append("--- Population Analysis ---")
            lines.append(f"CV > 20% \u2014 significant anisotropy detected.")
            lines.append(f"Median D = {pops['median']:.1f} nm")

            if 'large' in pops:
                lg = pops['large']
                lines.append(
                    f"  Large-D group: {lg['mean']:.1f} \u00b1 {lg['std']:.1f} nm "
                    f"(n={lg['n']}, range {lg['range'][0]:.1f}\u2013{lg['range'][1]:.1f} nm)")
            if 'small' in pops:
                sm = pops['small']
                lines.append(
                    f"  Small-D group: {sm['mean']:.1f} \u00b1 {sm['std']:.1f} nm "
                    f"(n={sm['n']}, range {sm['range'][0]:.1f}\u2013{sm['range'][1]:.1f} nm)")

            if 'angle_correlation' in pops:
                r = pops['angle_correlation']
                lines.append(f"  D vs sin(\u03b8) correlation: r = {r:.3f}")
                if r < -0.7:
                    lines.append("  \u2192 Strong negative correlation: microstrain likely dominates")
                elif r < -0.4:
                    lines.append("  \u2192 Moderate correlation: both strain and (hkl)-dependent broadening")
                else:
                    lines.append("  \u2192 Weak correlation: genuine anisotropic crystallite shape/disorder")

        if 'warning' in result:
            lines.append(f"\u26a0 {result['warning']}")

        lines.extend(["", "--- Method Notes ---", ResultsFormatter.NOTES['Scherrer']])
        return "\n".join(lines)

    @staticmethod
    def format_monshi_scherrer(result: Dict) -> str:
        if not result.get('success'):
            return f"Monshi-Scherrer Analysis: {result.get('message', 'Failed')}"

        lines = [
            "=" * 60,
            "MONSHI-SCHERRER ANALYSIS",
            "=" * 60,
            f"Shape factor K = {result['k_constant']}",
            f"Peaks used: {result['n_peaks']}",
            f"Plot: ln(β) vs ln(1/cos θ)",
            "-" * 60,
            f"Slope:     {result['slope']:.4f}  (expected ≈ 1.0, quality: {result['slope_quality']})",
            f"Intercept: {result['intercept']:.4f}  = ln(Kλ/L)",
            f"R²:        {result['r_squared']:.4f}",
            "-" * 60,
            f"Crystallite size: {result['crystallite_size_nm']:.1f} nm",
        ]
        if 'warning' in result:
            lines.append(f"⚠ {result['warning']}")

        lines.extend(["", "--- Method Notes ---", ResultsFormatter.NOTES['Monshi-Scherrer']])
        return "\n".join(lines)

    @staticmethod
    def format_williamson_hall(result: Dict) -> str:
        if not result.get('success'):
            return f"Williamson-Hall Analysis: {result.get('message', 'Failed')}"

        lines = [
            "=" * 60,
            "WILLIAMSON-HALL ANALYSIS (UDM)",
            "=" * 60,
            f"Shape factor K = {result['k_constant']}",
            f"Peaks used: {result['n_peaks']}",
            f"Plot: β cos θ vs 4 sin θ",
            "-" * 60,
            f"Slope (ε):     {result['linear_fit'].slope:.4e}",
            f"Intercept (Kλ/D): {result['linear_fit'].intercept:.4e}",
            f"R²:            {result['r_squared']:.4f}",
            "-" * 60,
        ]

        if result['crystallite_size_nm'] < float('inf'):
            lines.append(f"Crystallite size: {result['crystallite_size_nm']:.1f} nm")
        else:
            lines.append("Crystallite size: could not be determined (intercept ≤ 0)")
        lines.append(f"Microstrain: {result['microstrain_percent']:.4f}%")

        if 'warning' in result:
            lines.append(f"⚠ {result['warning']}")

        lines.extend(["", "--- Method Notes ---", ResultsFormatter.NOTES['Williamson-Hall']])
        return "\n".join(lines)

    @staticmethod
    def format_halder_wagner(result: Dict) -> str:
        if not result.get('success'):
            return f"Halder-Wagner Analysis: {result.get('message', 'Failed')}"

        lines = [
            "=" * 60,
            "HALDER-WAGNER ANALYSIS",
            "=" * 60,
            f"Shape factor K = {result['k_constant']}",
            f"Peaks used: {result['n_peaks']}",
            f"Plot: (β*/d*)² vs β*/(d*)²",
            f"      where β* = β cos θ/λ, d* = 2 sin θ/λ",
            "-" * 60,
            f"Slope (K/D):    {result['linear_fit'].slope:.4e}",
            f"Intercept (2ε)²: {result['linear_fit'].intercept:.4e}",
            f"R²:             {result['r_squared']:.4f}",
            "-" * 60,
        ]

        if result['crystallite_size_nm'] < float('inf'):
            lines.append(f"Crystallite size: {result['crystallite_size_nm']:.1f} nm")
        else:
            lines.append("Crystallite size: could not be determined (slope ≤ 0)")
        lines.append(f"Microstrain: {result['microstrain_percent']:.4f}%")

        if 'warning' in result:
            lines.append(f"⚠ {result['warning']}")

        lines.extend(["", "--- Method Notes ---", ResultsFormatter.NOTES['Halder-Wagner']])
        return "\n".join(lines)

    @staticmethod
    def format_ssp(result: Dict) -> str:
        if not result.get('success'):
            return f"SSP Analysis: {result.get('message', 'Failed')}"

        lines = [
            "=" * 60,
            "SIZE-STRAIN PLOT (SSP) ANALYSIS",
            "=" * 60,
            f"Shape factor K = {result['k_constant']}",
            f"Peaks used: {result['n_peaks']}",
            f"Plot: (d·β cos θ)² vs d²·β cos θ",
            "-" * 60,
            f"Slope (Kλ/D):    {result['linear_fit'].slope:.4e}",
            f"Intercept (ε/2)²: {result['linear_fit'].intercept:.4e}",
            f"R²:              {result['r_squared']:.4f}",
            "-" * 60,
        ]

        if result['crystallite_size_nm'] < float('inf'):
            lines.append(f"Crystallite size: {result['crystallite_size_nm']:.1f} nm")
        else:
            lines.append("Crystallite size: could not be determined (slope ≤ 0)")
        lines.append(f"Microstrain: {result['microstrain_percent']:.4f}%")

        if 'warning' in result:
            lines.append(f"⚠ {result['warning']}")

        lines.extend(["", "--- Method Notes ---", ResultsFormatter.NOTES['SSP']])
        return "\n".join(lines)

    @staticmethod
    def format_comparison(results: Dict[str, Dict]) -> str:
        lines = [
            "=" * 70,
            "METHOD COMPARISON",
            "=" * 70,
            f"{'Method':<20} {'D (nm)':>10} {'ε (%)':>10} {'R²':>8} {'Peaks':>6}",
            "-" * 70,
        ]

        for method_name, result in results.items():
            if not result.get('success'):
                lines.append(f"{method_name:<20} {'FAILED':>10}")
                continue

            size = result.get('crystallite_size_nm',
                              result.get('average_size_nm', float('nan')))
            size_str = f"{size:.1f}" if size < float('inf') and not np.isnan(size) else "N/A"

            strain = result.get('microstrain_percent', float('nan'))
            strain_str = f"{strain:.4f}" if not np.isnan(strain) else "N/A"

            r_sq = result.get('r_squared', float('nan'))
            r_sq_str = f"{r_sq:.4f}" if not np.isnan(r_sq) else "N/A"

            n_peaks = result.get('n_peaks', 0)
            lines.append(f"{method_name:<20} {size_str:>10} {strain_str:>10} {r_sq_str:>8} {n_peaks:>6}")

        lines.append("=" * 70)

        # Add interpretation guidance
        lines.extend([
            "",
            "--- Interpretation Guidance ---",
            "Scherrer gives a LOWER BOUND on crystallite size (ignores strain).",
            "W-H assumes Lorentzian profiles — typically overestimates strain.",
            "H-W/SSP assume Voigt profiles — usually the most reliable estimate.",
            "If D values vary strongly between methods, the material likely has",
            "anisotropic broadening, mixed broadening mechanisms, or defect",
            "structures that violate the assumptions of all isotropic methods.",
            "Consider: is the D vs 2θ plot flat? Is W-H R² > 0.9? Is the",
            "Monshi slope ≈ 1? If not, report ranges rather than single values.",
        ])

        return "\n".join(lines)


# =============================================================================
# CSV EXPORT
# =============================================================================

class ResultsExporter:
    """Export analysis results to CSV."""

    @staticmethod
    def export_peak_data(peaks: List[AnalysedPeak], file_path: str):
        """Export analysed peak data to CSV."""
        import csv
        with open(file_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                '2theta (deg)', 'theta (rad)', 'd-spacing (nm)',
                'FWHM_obs (deg)', 'FWHM_inst (deg)', 'beta_corrected (deg)',
                'beta_corrected (rad)', 'broadening_ratio',
                'resolution_limited', 'fit_R_squared', 'intensity'
            ])
            for p in peaks:
                writer.writerow([
                    f"{p.two_theta_deg:.4f}",
                    f"{p.theta_rad:.6f}",
                    f"{p.d_spacing_nm:.5f}",
                    f"{p.fwhm_obs_deg:.5f}",
                    f"{p.fwhm_inst_deg:.5f}",
                    f"{p.beta_corrected_deg:.5f}",
                    f"{p.beta_corrected_rad:.7f}",
                    f"{p.broadening_ratio:.4f}",
                    p.resolution_limited,
                    f"{p.fit_quality:.4f}",
                    f"{p.intensity:.2f}",
                ])

    @staticmethod
    def export_summary(results: Dict[str, Dict], file_path: str):
        """Export method comparison summary to CSV."""
        import csv
        with open(file_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                'Method', 'D (nm)', 'D_std (nm)', 'Microstrain (%)',
                'R_squared', 'K_constant', 'N_peaks', 'Warnings'
            ])
            for method_name, result in results.items():
                if not result.get('success'):
                    writer.writerow([method_name, 'FAILED', '', '', '', '', '',
                                     result.get('message', '')])
                    continue

                size = result.get('crystallite_size_nm',
                                  result.get('average_size_nm', ''))
                std = result.get('std_size_nm', '')
                strain = result.get('microstrain_percent', '')
                r_sq = result.get('r_squared', '')
                k = result.get('k_constant', '')
                n = result.get('n_peaks', '')
                warn = result.get('warning', '')

                writer.writerow([
                    method_name,
                    f"{size:.2f}" if isinstance(size, (int, float)) and size < float('inf') else '',
                    f"{std:.2f}" if isinstance(std, (int, float)) else '',
                    f"{strain:.4f}" if isinstance(strain, (int, float)) else '',
                    f"{r_sq:.4f}" if isinstance(r_sq, (int, float)) else '',
                    k, n, warn
                ])


# =============================================================================
# CONVENIENCE: RUN ALL METHODS
# =============================================================================

def run_all_methods(peaks: List[AnalysedPeak],
                    wavelength_nm: float,
                    k_constant: float = 0.94) -> Dict[str, Dict]:
    """
    Run all five analysis methods on the same set of peaks.

    Args:
        peaks: List of AnalysedPeak objects.
        wavelength_nm: Kα1 wavelength in nm.
        k_constant: Scherrer shape factor.

    Returns:
        Dict mapping method name → result dict.
    """
    results = {}

    # Scherrer
    scherrer = ScherrerAnalyzer(wavelength_nm)
    results['Scherrer'] = scherrer.analyse(peaks, k_constant)

    # Monshi-Scherrer
    monshi = MonshiScherrerAnalyzer(wavelength_nm)
    results['Monshi-Scherrer'] = monshi.analyse(peaks, k_constant)

    # Williamson-Hall
    wh = WilliamsonHallAnalyzer(wavelength_nm)
    results['Williamson-Hall'] = wh.analyse(peaks, k_constant)

    # Halder-Wagner
    hw = HalderWagnerAnalyzer(wavelength_nm)
    results['Halder-Wagner'] = hw.analyse(peaks, k_constant)

    # SSP
    ssp = SSPAnalyzer(wavelength_nm)
    results['SSP'] = ssp.analyse(peaks, k_constant)

    return results

In [ ]:
"""
Völva 4.0 — Plotting Module
============================

Publication-quality diagnostic and analysis plots.
All plotting logic lives here; the GUI just calls these functions.

Every plot function takes a matplotlib Figure (or creates one) and
returns the Figure, so they work both embedded in the tkinter GUI
and standalone for export.

Colour scheme: colorblind-accessible (Okabe-Ito palette).
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as ticker
from typing import Dict, List, Optional, Tuple, Any

# Try to import engine types for type hints
try:
    from volva_engine import (
        AnalysedPeak, PeakFitResult, LinearFitResult,
        InstrumentalCorrection, BackgroundProcessor,
        PeakFitter, WAVELENGTHS
    )
except ImportError:
    pass  # Will work via globals in notebook context


# =============================================================================
# COLOUR PALETTE — Okabe-Ito, colorblind-safe
# =============================================================================

C = {
    'blue':       '#0072B2',
    'orange':     '#E69F00',
    'green':      '#009E73',
    'vermillion': '#D55E00',
    'sky':        '#56B4E9',
    'pink':       '#CC79A7',
    'yellow':     '#F0E442',
    'black':      '#000000',
    'grey':       '#999999',
    'lightgrey':  '#DDDDDD',
    'bg_curve':   '#CC0000',
}

# Per-method colour assignments
METHOD_COLOURS = {
    'Scherrer':        C['blue'],
    'Monshi-Scherrer': C['vermillion'],
    'Williamson-Hall': C['green'],
    'Halder-Wagner':   C['orange'],
    'SSP':             C['pink'],
}


# =============================================================================
# COMMON FORMATTING
# =============================================================================

def _style_axis(ax, xlabel=None, ylabel=None, title=None, grid=True):
    """Apply consistent formatting to an axis."""
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=10)
    if title:
        ax.set_title(title, fontsize=11, fontweight='bold')
    if grid:
        ax.grid(True, alpha=0.2, linewidth=0.5)
    ax.tick_params(labelsize=9)


def _annotate_box(ax, text, loc='upper right', fontsize=9):
    """Place an annotation box on an axis."""
    loc_map = {
        'upper right': (0.97, 0.97, 'right', 'top'),
        'upper left':  (0.03, 0.97, 'left',  'top'),
        'lower right': (0.97, 0.03, 'right', 'bottom'),
        'lower left':  (0.03, 0.03, 'left',  'bottom'),
    }
    x, y, ha, va = loc_map.get(loc, loc_map['upper right'])
    ax.text(x, y, text, transform=ax.transAxes, fontsize=fontsize,
            ha=ha, va=va,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='wheat',
                      alpha=0.85, edgecolor='#999999'))


# =============================================================================
# PLOT 1: XRD PATTERN WITH BACKGROUND
# =============================================================================

def plot_xrd_with_background(fig: Figure,
                              two_theta: np.ndarray,
                              intensity: np.ndarray,
                              background: np.ndarray,
                              subtracted: np.ndarray,
                              filename: str = "",
                              std_two_theta: Optional[np.ndarray] = None,
                              std_intensity: Optional[np.ndarray] = None,
                              std_filename: str = "") -> Figure:
    """
    XRD pattern showing raw data, background, and subtracted result.

    Three panels:
        Top:    Raw data + background curve
        Middle: Background-subtracted data
        Bottom: Standard (if provided)
    """
    fig.clear()

    n_panels = 3 if std_two_theta is not None else 2
    heights = [3, 2, 2] if n_panels == 3 else [3, 2]
    gs = GridSpec(n_panels, 1, figure=fig, height_ratios=heights, hspace=0.08)

    # --- Raw + background ---
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(two_theta, intensity, color=C['black'], linewidth=0.5,
             label='Raw data', zorder=1)
    ax1.plot(two_theta, background, color=C['bg_curve'], linewidth=1.2,
             alpha=0.8, label='Background (SNIP)', zorder=2)
    ax1.fill_between(two_theta, 0, background, color=C['bg_curve'],
                      alpha=0.05, zorder=0)
    _style_axis(ax1, ylabel='Intensity (counts)',
                title=f'XRD Pattern — {filename}' if filename else 'XRD Pattern')
    ax1.legend(loc='upper right', fontsize=8, framealpha=0.8)
    ax1.tick_params(labelbottom=False)
    ax1.set_xlim(two_theta[0], two_theta[-1])

    # --- Subtracted ---
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.plot(two_theta, subtracted, color=C['blue'], linewidth=0.5,
             label='Background-subtracted')
    ax2.axhline(0, color='k', linewidth=0.3)
    _style_axis(ax2, ylabel='Intensity (counts)')
    ax2.legend(loc='upper right', fontsize=8, framealpha=0.8)

    if std_two_theta is None:
        ax2.set_xlabel(r'$2\theta$ / °', fontsize=10)
    else:
        ax2.tick_params(labelbottom=False)

    # --- Standard ---
    if std_two_theta is not None:
        ax3 = fig.add_subplot(gs[2], sharex=ax1)
        ax3.plot(std_two_theta, std_intensity, color=C['green'], linewidth=0.5,
                 label=f'Standard: {std_filename}')
        _style_axis(ax3, xlabel=r'$2\theta$ / °', ylabel='Intensity (counts)')
        ax3.legend(loc='upper right', fontsize=8, framealpha=0.8)

    fig.align_ylabels()
    return fig


# =============================================================================
# PLOT 2: LaB₆ STANDARD DIAGNOSTIC
# =============================================================================

def plot_standard_diagnostic(fig: Figure,
                              two_theta: np.ndarray,
                              intensity: np.ndarray,
                              inst_correction,
                              filename: str = "") -> Figure:
    """
    LaB₆ standard diagnostic plot.

    Two panels:
        Top:    Standard pattern with each fitted peak overlaid,
                FWHM and R² annotated per peak
        Bottom: Instrumental FWHM vs 2θ interpolation curve
    """
    fig.clear()
    gs = GridSpec(2, 1, figure=fig, height_ratios=[3, 2], hspace=0.25)

    # --- Top: pattern with fits ---
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(two_theta, intensity, color=C['green'], linewidth=0.5,
             alpha=0.7, label='Standard data', zorder=1)

    for peak_info in inst_correction.standard_peaks:
        fit = peak_info.get('fit_result')
        if fit is not None and fit.fit_x is not None:
            # Estimate background level at peak position for offset
            peak_pos = fit.center_deg
            nearby_mask = (two_theta >= peak_pos - 2.0) & (two_theta <= peak_pos + 2.0)
            if np.any(nearby_mask):
                bg_level = np.percentile(intensity[nearby_mask], 10)
            else:
                bg_level = 0

            ax1.plot(fit.fit_x, fit.fit_y + bg_level,
                     color=C['vermillion'], linewidth=1.8, alpha=0.9, zorder=3)

            # Annotate with FWHM
            ax1.annotate(
                f"{peak_info['fwhm']:.3f}°",
                xy=(peak_pos, np.max(fit.fit_y) + bg_level),
                xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=7, color=C['vermillion'],
                fontweight='bold',
                arrowprops=dict(arrowstyle='-', color=C['vermillion'],
                                linewidth=0.5)
            )

    _style_axis(ax1, ylabel='Intensity (counts)',
                title=f'LaB₆ Standard — {filename}' if filename else
                       'LaB₆ Standard — Peak Fitting')
    ax1.legend(loc='upper right', fontsize=8, framealpha=0.8)
    ax1.set_xlim(two_theta[0], two_theta[-1])

    # Add fit legend entry manually
    from matplotlib.lines import Line2D
    handles = ax1.get_legend_handles_labels()[0]
    handles.append(Line2D([0], [0], color=C['vermillion'], linewidth=1.8,
                          label='Pseudo-Voigt fits'))
    ax1.legend(handles=handles, loc='upper right', fontsize=8, framealpha=0.8)

    # --- Bottom: FWHM vs 2θ ---
    ax2 = fig.add_subplot(gs[1])

    peak_positions = [p['two_theta'] for p in inst_correction.standard_peaks]
    peak_fwhms = [p['fwhm'] for p in inst_correction.standard_peaks]
    peak_rsq = [p['r_squared'] for p in inst_correction.standard_peaks]

    # Colour points by R²
    scatter = ax2.scatter(peak_positions, peak_fwhms, c=peak_rsq,
                           cmap='RdYlGn', vmin=0.7, vmax=1.0,
                           s=70, edgecolors='k', linewidths=0.5, zorder=3)
    cbar = fig.colorbar(scatter, ax=ax2, shrink=0.8, pad=0.02)
    cbar.set_label('Fit R²', fontsize=9)

    # Interpolation curve (always shown for reference)
    x_interp = np.linspace(two_theta[0], two_theta[-1], 500)

    if inst_correction.caglioti_fitted:
        # Caglioti curve — the primary model
        theta_rad = np.deg2rad(x_interp / 2.0)
        tan_theta = np.tan(theta_rad)
        U = inst_correction.caglioti_U
        V = inst_correction.caglioti_V
        W = inst_correction.caglioti_W
        fwhm_sq = U * tan_theta ** 2 + V * tan_theta + W
        y_caglioti = np.sqrt(np.maximum(fwhm_sq, 0))
        ax2.plot(x_interp, y_caglioti, color=C['vermillion'], linewidth=2.0,
                 alpha=0.9, label='Caglioti fit', zorder=4)

        # Also show linear interpolation as faint reference
        y_linear = [float(np.interp(x, np.array(peak_positions), np.array(peak_fwhms)))
                     if peak_positions[0] <= x <= peak_positions[-1]
                     else (peak_fwhms[0] if x < peak_positions[0] else peak_fwhms[-1])
                     for x in x_interp]
        ax2.plot(x_interp, y_linear, color=C['sky'], linewidth=1.0,
                 alpha=0.4, linestyle=':', label='Linear interp.', zorder=2)
    else:
        # No Caglioti — just show interpolation
        y_interp = [inst_correction.get_fwhm_at(x) for x in x_interp]
        ax2.plot(x_interp, y_interp, color=C['sky'], linewidth=1.5,
                 alpha=0.8, label='Linear interp.', zorder=2)

    # Annotate each point with R²
    for pos, fwhm, rsq in zip(peak_positions, peak_fwhms, peak_rsq):
        ax2.annotate(f'{rsq:.3f}', (pos, fwhm),
                     xytext=(0, -14), textcoords='offset points',
                     ha='center', fontsize=6.5, color=C['grey'])

    _style_axis(ax2, xlabel=r'$2\theta$ / °',
                ylabel='Instrumental FWHM (°)',
                title='Instrumental Broadening Profile')
    ax2.legend(loc='upper left', fontsize=8, framealpha=0.8)
    ax2.set_xlim(two_theta[0], two_theta[-1])

    # Annotation box
    n_peaks = len(peak_positions)
    mean_fwhm = np.mean(peak_fwhms)
    ann_lines = [f'{n_peaks} peaks accepted', f'Mean FWHM = {mean_fwhm:.4f}\u00b0']

    if inst_correction.caglioti_fitted:
        ann_lines.extend([
            '',
            'FWHM\u00b2 = U tan\u00b2\u03b8 + V tan\u03b8 + W',
            f'U = {inst_correction.caglioti_U:+.6f}',
            f'V = {inst_correction.caglioti_V:+.6f}',
            f'W = {inst_correction.caglioti_W:+.6f}',
            f'R\u00b2 = {inst_correction.caglioti_r_squared:.4f}',
        ])

    _annotate_box(ax2, '\n'.join(ann_lines), loc='upper right', fontsize=7.5)

    fig.align_ylabels()
    return fig


# =============================================================================
# PLOT 3: XRD WITH FITTED PEAKS AND RESIDUALS
# =============================================================================

def plot_xrd_with_fits(fig: Figure,
                       two_theta: np.ndarray,
                       intensity: np.ndarray,
                       background: np.ndarray,
                       fit_results: list,
                       analysed_peaks: list,
                       filename: str = "",
                       inst_correction=None) -> Figure:
    """
    XRD pattern with fitted peak profiles and Rietveld-style residual.

    Three panels:
        Top:    Raw data + composite fit envelope + individual peak curves
        Middle: Residual (observed − fitted) in fit windows
        Bottom: Instrumental FWHM markers on the fit positions (if available)
    """
    fig.clear()

    has_inst = inst_correction is not None and inst_correction.is_loaded
    n_panels = 3 if has_inst else 2
    heights = [4, 1.2, 1.2] if n_panels == 3 else [4, 1.2]
    gs = GridSpec(n_panels, 1, figure=fig, height_ratios=heights, hspace=0.08)

    # --- Main pattern ---
    ax_main = fig.add_subplot(gs[0])
    ax_main.plot(two_theta, intensity, color=C['black'], linewidth=0.4,
                 alpha=0.6, label='Observed', zorder=1)
    ax_main.plot(two_theta, background, color=C['bg_curve'], linewidth=0.8,
                 alpha=0.4, linestyle='--', label='Background', zorder=1)

    # Plot each fitted peak
    successful_fits = [f for f in fit_results if f.success and f.fit_x is not None]
    for i, fit in enumerate(successful_fits):
        # Offset fit by background at that position
        bg_at_fit = np.interp(fit.fit_x, two_theta, background)

        colour = C['orange']
        ax_main.plot(fit.fit_x, fit.fit_y + bg_at_fit,
                     color=colour, linewidth=1.8, alpha=0.85, zorder=3)

        # Fill under fit curve
        ax_main.fill_between(fit.fit_x, bg_at_fit, fit.fit_y + bg_at_fit,
                              color=colour, alpha=0.08, zorder=2)

    # Peak markers
    if analysed_peaks:
        # Measurable peaks
        meas = [p for p in analysed_peaks if not p.resolution_limited]
        if meas:
            ax_main.scatter([p.two_theta_deg for p in meas],
                           [np.interp(p.two_theta_deg, two_theta, intensity) for p in meas],
                           color=C['blue'], marker='v', s=45, zorder=5,
                           label=f'Fitted peaks ({len(meas)})', edgecolors='k',
                           linewidths=0.3)

        # Resolution-limited peaks
        rl = [p for p in analysed_peaks if p.resolution_limited]
        if rl:
            ax_main.scatter([p.two_theta_deg for p in rl],
                           [np.interp(p.two_theta_deg, two_theta, intensity) for p in rl],
                           color='none', edgecolors='red', marker='o', s=100,
                           linewidths=1.5, zorder=6,
                           label=f'Resolution limited ({len(rl)})')

    _style_axis(ax_main, ylabel='Intensity (counts)',
                title=f'Peak Fitting — {filename}' if filename else 'Peak Fitting')
    ax_main.legend(loc='upper right', fontsize=7, framealpha=0.8, ncol=2)
    ax_main.tick_params(labelbottom=False)
    ax_main.set_xlim(two_theta[0], two_theta[-1])

    # --- Residual panel ---
    ax_res = fig.add_subplot(gs[1], sharex=ax_main)

    for fit in successful_fits:
        if fit.data_x is not None and fit.residuals is not None:
            ax_res.bar(fit.data_x, fit.residuals, width=np.median(np.diff(two_theta)),
                       color=C['grey'], alpha=0.6, edgecolor='none')

    ax_res.axhline(0, color='k', linewidth=0.5)
    _style_axis(ax_res, ylabel='Residual', grid=False)
    ax_res.tick_params(labelbottom=(not has_inst))
    if not has_inst:
        ax_res.set_xlabel(r'$2\theta$ / °', fontsize=10)

    # --- Instrumental FWHM panel ---
    if has_inst:
        ax_inst = fig.add_subplot(gs[2], sharex=ax_main)

        # Draw instrumental FWHM as horizontal bars at each peak position
        for peak in analysed_peaks:
            pos = peak.two_theta_deg
            fwhm_inst = peak.fwhm_inst_deg
            fwhm_obs = peak.fwhm_obs_deg
            beta_corr = peak.beta_corrected_deg

            # Observed FWHM bar (full width)
            ax_inst.barh(0.6, fwhm_obs, height=0.25, left=pos - fwhm_obs / 2,
                         color=C['blue'], alpha=0.4, edgecolor=C['blue'], linewidth=0.5)

            # Instrumental FWHM bar
            ax_inst.barh(0.3, fwhm_inst, height=0.25, left=pos - fwhm_inst / 2,
                         color=C['green'], alpha=0.6, edgecolor=C['green'], linewidth=0.5)

            # Corrected FWHM bar
            ax_inst.barh(0.0, beta_corr, height=0.25, left=pos - beta_corr / 2,
                         color=C['orange'], alpha=0.6, edgecolor=C['orange'], linewidth=0.5)

        ax_inst.set_yticks([0.0, 0.3, 0.6])
        ax_inst.set_yticklabels([r'$\beta_{corr}$', r'$\beta_{inst}$', r'$\beta_{obs}$'],
                                 fontsize=8)
        ax_inst.set_ylim(-0.15, 0.9)
        _style_axis(ax_inst, xlabel=r'$2\theta$ / °', grid=False)

    fig.align_ylabels()
    return fig


# =============================================================================
# PLOT 4: PER-PEAK FIT INSPECTOR
# =============================================================================

def plot_peak_inspector(fig: Figure,
                        fit_result,
                        peak: Optional[object] = None,
                        peak_index: int = 0,
                        wavelength_info: Optional[Dict] = None) -> Figure:
    """
    Detailed view of a single peak fit.

    Two panels:
        Top:    Data + composite fit + individual Kα components (if doublet)
        Bottom: Residual

    Annotation box with FWHM, R², χ², fraction, instrumental FWHM.
    """
    fig.clear()

    if not fit_result.success or fit_result.data_x is None:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"Peak {peak_index + 1}: {fit_result.message}",
                ha='center', va='center', fontsize=12, color=C['grey'],
                transform=ax.transAxes)
        ax.set_axis_off()
        return fig

    gs = GridSpec(2, 1, figure=fig, height_ratios=[3, 1], hspace=0.08)

    # --- Fit panel ---
    ax_fit = fig.add_subplot(gs[0])

    # Data points
    ax_fit.scatter(fit_result.data_x, fit_result.data_y,
                   color=C['blue'], s=12, zorder=3, label='Observed',
                   alpha=0.7, edgecolors='none')

    # Composite fit
    ax_fit.plot(fit_result.fit_x, fit_result.fit_y,
                color=C['vermillion'], linewidth=2.0, zorder=4,
                label='Composite fit')

    # If doublet, show individual Kα components
    if fit_result.doublet_fitted and wavelength_info is not None:
        try:
            from lmfit.models import PseudoVoigtModel

            # We can reconstruct the components from the fit parameters
            # The fit_result stores the composite, so we extract Kα1 and Kα2
            # from the known parameter relationships
            model1 = PseudoVoigtModel(prefix='k1_')

            # Kα1 component
            ka1_y = model1.eval(
                x=fit_result.fit_x,
                k1_amplitude=fit_result.amplitude,
                k1_center=fit_result.center_deg,
                k1_sigma=fit_result.sigma,
                k1_fraction=fit_result.fraction
            )
            ax_fit.plot(fit_result.fit_x, ka1_y,
                        color=C['sky'], linewidth=1.0, linestyle='--',
                        alpha=0.8, zorder=2, label='Kα₁')

            # Kα2 component
            ka1 = wavelength_info.get('ka1', 0.154056)
            ka2 = wavelength_info.get('ka2', 0.154439)
            ratio = wavelength_info.get('ratio', 0.5)
            theta_rad = np.deg2rad(fit_result.center_deg / 2.0)
            delta_2theta = 2 * np.rad2deg(
                np.tan(theta_rad) * (ka2 - ka1) / ka1)

            ka2_y = model1.eval(
                x=fit_result.fit_x,
                k1_amplitude=fit_result.amplitude * ratio,
                k1_center=fit_result.center_deg + delta_2theta,
                k1_sigma=fit_result.sigma,
                k1_fraction=fit_result.fraction
            )
            ax_fit.plot(fit_result.fit_x, ka2_y,
                        color=C['pink'], linewidth=1.0, linestyle='--',
                        alpha=0.8, zorder=2, label='Kα₂')
        except Exception:
            pass  # Silently skip if reconstruction fails

    # FWHM visualisation: horizontal line at half-maximum
    half_max = np.max(fit_result.fit_y) / 2.0
    ax_fit.axhline(half_max, color=C['grey'], linewidth=0.5, linestyle=':',
                    alpha=0.5)

    # Mark FWHM span
    above_half = fit_result.fit_x[fit_result.fit_y >= half_max]
    if len(above_half) >= 2:
        fwhm_left = above_half[0]
        fwhm_right = above_half[-1]
        ax_fit.annotate('', xy=(fwhm_left, half_max), xytext=(fwhm_right, half_max),
                        arrowprops=dict(arrowstyle='<->', color=C['orange'],
                                        linewidth=1.5))
        ax_fit.text((fwhm_left + fwhm_right) / 2, half_max * 1.08,
                    f'FWHM = {fit_result.fwhm_deg:.4f}°',
                    ha='center', fontsize=8, color=C['orange'], fontweight='bold')

    _style_axis(ax_fit, ylabel='Intensity (bg-subtracted)',
                title=f'Peak {peak_index + 1}: 2θ = {fit_result.center_deg:.3f}°')
    ax_fit.legend(loc='upper right', fontsize=7, framealpha=0.8)
    ax_fit.tick_params(labelbottom=False)

    # Annotation box
    annotation_lines = [
        f"Centre: {fit_result.center_deg:.4f}°",
        f"FWHM: {fit_result.fwhm_deg:.4f}°",
        f"R² = {fit_result.r_squared:.4f}",
        f"Red. χ² = {fit_result.reduced_chi_sq:.2f}",
        f"η (Lorentzian): {fit_result.fraction:.3f}",
    ]
    if fit_result.doublet_fitted:
        annotation_lines.append("Kα₁/Kα₂ doublet: Yes")
    else:
        annotation_lines.append("Kα₁/Kα₂ doublet: No")

    if peak is not None:
        annotation_lines.extend([
            f"β_inst: {peak.fwhm_inst_deg:.4f}°",
            f"β_corr: {peak.beta_corrected_deg:.4f}°",
            f"Broadening ratio: {peak.broadening_ratio:.2f}",
        ])
        if peak.resolution_limited:
            annotation_lines.append("⚠ RESOLUTION LIMITED")

    _annotate_box(ax_fit, '\n'.join(annotation_lines),
                  loc='upper left', fontsize=8)

    # --- Residual panel ---
    ax_res = fig.add_subplot(gs[1], sharex=ax_fit)

    if fit_result.residuals is not None:
        ax_res.bar(fit_result.data_x, fit_result.residuals,
                   width=np.median(np.diff(fit_result.data_x)) * 0.8,
                   color=C['grey'], alpha=0.7, edgecolor='none')
        ax_res.axhline(0, color='k', linewidth=0.5)

        # ±2σ lines
        res_std = np.std(fit_result.residuals)
        ax_res.axhline(2 * res_std, color=C['vermillion'], linewidth=0.5,
                        linestyle='--', alpha=0.5)
        ax_res.axhline(-2 * res_std, color=C['vermillion'], linewidth=0.5,
                        linestyle='--', alpha=0.5)

    _style_axis(ax_res, xlabel=r'$2\theta$ / °', ylabel='Residual', grid=False)

    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fig.tight_layout()
    return fig


# =============================================================================
# PLOT 5: SCHERRER D vs 2θ (ANISOTROPY CHECK)
# =============================================================================

def plot_scherrer_vs_2theta(fig: Figure,
                             scherrer_result: Dict,
                             filename: str = "") -> Figure:
    """
    Scherrer crystallite size vs 2θ — the key anisotropy diagnostic.

    Single panel with:
        - D values as large points (colour-coded by resolution-limited status)
        - Mean D ± σ as horizontal band
        - Per-peak annotations
    """
    fig.clear()

    if not scherrer_result.get('success'):
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"Scherrer: {scherrer_result.get('message', 'Failed')}",
                ha='center', va='center', fontsize=12, color=C['grey'],
                transform=ax.transAxes)
        ax.set_axis_off()
        return fig

    ax = fig.add_subplot(111)

    results = scherrer_result['results']
    two_thetas = [r['two_theta'] for r in results]
    sizes = [r['size_nm'] for r in results]
    res_limited = [r['resolution_limited'] for r in results]

    # Separate measurable and resolution-limited
    meas_x = [t for t, rl in zip(two_thetas, res_limited) if not rl]
    meas_y = [s for s, rl in zip(sizes, res_limited) if not rl]
    rl_x = [t for t, rl in zip(two_thetas, res_limited) if rl]
    rl_y = [s for s, rl in zip(sizes, res_limited) if rl]

    # Plot measurable peaks
    if meas_x:
        ax.scatter(meas_x, meas_y, s=100, color=C['blue'], edgecolors='k',
                   linewidths=0.5, zorder=3, label='Measurable')

    # Plot resolution-limited peaks
    if rl_x:
        ax.scatter(rl_x, rl_y, s=100, color=C['lightgrey'], edgecolors='red',
                   linewidths=1.5, zorder=3, marker='^',
                   label='Resolution limited (lower bound)')

    # Mean ± σ band
    valid_sizes = [s for s, rl in zip(sizes, res_limited)
                   if not rl and not np.isnan(s)]
    if valid_sizes:
        mean_d = np.mean(valid_sizes)
        std_d = np.std(valid_sizes)
        ax.axhspan(mean_d - std_d, mean_d + std_d, color=C['sky'],
                    alpha=0.15, zorder=0)
        ax.axhline(mean_d, color=C['blue'], linewidth=1.0, linestyle='--',
                    alpha=0.6, zorder=1,
                    label=f'Mean = {mean_d:.1f} ± {std_d:.1f} nm')

    # Per-peak annotations
    for t, s, rl in zip(two_thetas, sizes, res_limited):
        if not np.isnan(s):
            offset = 8 if not rl else -15
            ax.annotate(f'{s:.0f}', (t, s), textcoords='offset points',
                        xytext=(0, offset), ha='center', fontsize=9,
                        fontweight='bold', color=C['vermillion'] if rl else C['black'])

    _style_axis(ax, xlabel=r'$2\theta$ / °',
                ylabel='Crystallite size D (nm)',
                title=f'Scherrer Size vs 2θ — Anisotropy Check'
                      + (f'\n{filename}' if filename else ''))
    ax.legend(loc='best', fontsize=8, framealpha=0.8)

    # Check for anisotropy and annotate
    if valid_sizes and len(valid_sizes) >= 3:
        cv = std_d / mean_d * 100 if mean_d > 0 else 0
        if cv > 20:
            quality = "Strong anisotropy"
        elif cv > 10:
            quality = "Moderate anisotropy"
        else:
            quality = "Isotropic (uniform)"
        _annotate_box(ax,
                      f'CV = {cv:.1f}%\n{quality}\n'
                      f'K = {scherrer_result["k_constant"]}',
                      loc='upper right', fontsize=9)

    return fig


# =============================================================================
# PLOT 6: LINEAR METHOD WITH CONFIDENCE BANDS
# =============================================================================

def plot_linear_method(fig: Figure,
                       result: Dict,
                       method_name: str) -> Figure:
    """
    Linear regression plot for W-H, H-W, SSP, or Monshi-Scherrer
    with 95% confidence and prediction bands.
    """
    fig.clear()

    if not result.get('success'):
        ax = fig.add_subplot(111)
        status = result.get('message', 'Failed')
        ax.text(0.5, 0.5, f"{method_name}\n{status}",
                ha='center', va='center', fontsize=14, color=C['grey'],
                transform=ax.transAxes)
        ax.set_axis_off()
        return fig

    lf = result.get('linear_fit')
    if lf is None or lf.x_data is None:
        return fig

    ax = fig.add_subplot(111)

    x = lf.x_data
    y = lf.y_data
    n = len(x)

    # Linear fit
    y_pred = lf.slope * x + lf.intercept
    x_fit = np.linspace(x.min() * 0.9, x.max() * 1.1, 200)
    y_fit = lf.slope * x_fit + lf.intercept

    # 95% confidence bands
    x_mean = np.mean(x)
    ss_x = np.sum((x - x_mean) ** 2)
    se_y = np.sqrt(np.sum((y - y_pred) ** 2) / max(n - 2, 1))

    # Confidence band (for the regression line)
    from scipy import stats
    t_val = stats.t.ppf(0.975, max(n - 2, 1))
    se_conf = se_y * np.sqrt(1.0 / n + (x_fit - x_mean) ** 2 / ss_x)
    y_conf_upper = y_fit + t_val * se_conf
    y_conf_lower = y_fit - t_val * se_conf

    # Prediction band (for individual observations)
    se_pred = se_y * np.sqrt(1.0 + 1.0 / n + (x_fit - x_mean) ** 2 / ss_x)
    y_pred_upper = y_fit + t_val * se_pred
    y_pred_lower = y_fit - t_val * se_pred

    # Plot
    colour = METHOD_COLOURS.get(method_name, C['blue'])

    # Prediction band
    ax.fill_between(x_fit, y_pred_lower, y_pred_upper,
                     color=colour, alpha=0.06, label='95% prediction')
    # Confidence band
    ax.fill_between(x_fit, y_conf_lower, y_conf_upper,
                     color=colour, alpha=0.18, label='95% confidence')
    # Fit line
    ax.plot(x_fit, y_fit, color=colour, linewidth=2.0,
            linestyle='--', zorder=4, label='Linear fit')
    # Data points
    ax.scatter(x, y, color=colour, s=80, edgecolors='k', linewidths=0.5,
               zorder=5, label='Data')

    # Axis labels by method
    labels = {
        'Williamson-Hall': {
            'x': r'$4\,\sin\theta$',
            'y': r'$\beta\,\cos\theta$ (rad)',
            'title': 'Williamson-Hall Plot (UDM)',
        },
        'Halder-Wagner': {
            'x': r'$\beta^{\,*}\,/\,(d^{\,*})^2$',
            'y': r'$(\beta^{\,*}\,/\,d^{\,*})^2$',
            'title': 'Halder-Wagner Plot',
        },
        'SSP': {
            'x': r'$d^2_{hkl}\;\beta\,\cos\theta$',
            'y': r'$(d_{hkl}\;\beta\,\cos\theta)^2$',
            'title': 'Size-Strain Plot (SSP)',
        },
        'Monshi-Scherrer': {
            'x': r'$\ln(1\,/\,\cos\theta)$',
            'y': r'$\ln\,\beta$ (rad)',
            'title': 'Monshi-Scherrer Plot',
        },
    }
    lab = labels.get(method_name, {'x': 'x', 'y': 'y', 'title': method_name})

    _style_axis(ax, xlabel=lab['x'], ylabel=lab['y'], title=lab['title'])
    ax.legend(loc='lower right', fontsize=8, framealpha=0.8)

    # Annotation box with results
    size = result.get('crystallite_size_nm', result.get('average_size_nm', None))
    strain = result.get('microstrain_percent', None)

    lines = [f'R² = {lf.r_squared:.4f}', f'n = {n} peaks']
    if size is not None and size < float('inf'):
        lines.append(f'D = {size:.1f} nm')
    if strain is not None:
        lines.append(f'ε = {strain:.4f}%')
    if method_name == 'Monshi-Scherrer':
        lines.append(f'slope = {lf.slope:.3f} (expect ≈ 1.0)')
        quality = result.get('slope_quality', '')
        lines.append(f'slope quality: {quality}')

    _annotate_box(ax, '\n'.join(lines), loc='upper left', fontsize=9)

    fig.tight_layout()
    return fig


# =============================================================================
# PLOT 7: METHOD COMPARISON
# =============================================================================

def plot_method_comparison(fig: Figure,
                           results: Dict[str, Dict],
                           filename: str = "") -> Figure:
    """
    Side-by-side bar charts comparing all methods.
    """
    fig.clear()

    gs = GridSpec(1, 2, figure=fig, wspace=0.3)
    ax_size = fig.add_subplot(gs[0])
    ax_strain = fig.add_subplot(gs[1])

    # Abbreviated labels for x-axis
    ABBREV = {
        'Scherrer': 'Scherrer',
        'Monshi-Scherrer': 'Monshi',
        'Williamson-Hall': 'W-H',
        'Halder-Wagner': 'H-W',
        'SSP': 'SSP',
    }

    methods = []
    labels = []
    sizes = []
    size_errs = []
    strains = []
    r_squareds = []
    colours = []

    for name, res in results.items():
        if not res.get('success'):
            continue

        methods.append(name)
        labels.append(ABBREV.get(name, name))

        size = res.get('crystallite_size_nm',
                       res.get('average_size_nm', float('nan')))
        if size == float('inf'):
            size = float('nan')
        sizes.append(size)
        size_errs.append(res.get('std_size_nm', 0))

        strains.append(res.get('microstrain_percent', float('nan')))
        r_squareds.append(res.get('r_squared', float('nan')))
        colours.append(METHOD_COLOURS.get(name, C['blue']))

    if not methods:
        ax_size.text(0.5, 0.5, "No results", ha='center', va='center',
                     fontsize=14, color=C['grey'], transform=ax_size.transAxes)
        return fig

    x = np.arange(len(methods))
    bar_width = 0.6

    # --- Size bars ---
    for i, (s, e, col) in enumerate(zip(sizes, size_errs, colours)):
        if not np.isnan(s):
            err = e if e > 0 else None
            ax_size.bar(x[i], s, bar_width, yerr=err,
                        color=col, alpha=0.8, capsize=4,
                        edgecolor='k', linewidth=0.5)
            y_text = s + (e if e > 0 else s * 0.02)
            ax_size.text(x[i], y_text, f'{s:.1f}',
                         ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax_size.set_xticks(x)
    ax_size.set_xticklabels(labels, fontsize=9)
    _style_axis(ax_size, ylabel='Crystallite size D (nm)')

    # --- Strain bars ---
    has_strain = False
    for i, (strain, rsq, col) in enumerate(zip(strains, r_squareds, colours)):
        if not np.isnan(strain):
            has_strain = True
            ax_strain.bar(x[i], strain, bar_width,
                          color=col, alpha=0.8,
                          edgecolor='k', linewidth=0.5)
            label = f'{strain:.3f}%'
            if not np.isnan(rsq):
                label += f'\nR\u00b2={rsq:.3f}'
            ax_strain.text(x[i], strain * 1.02,
                           label, ha='center', va='bottom',
                           fontsize=7.5, fontweight='bold')

    if not has_strain:
        ax_strain.text(0.5, 0.5, 'No strain data\n(Scherrer only)',
                       ha='center', va='center', fontsize=10,
                       color=C['grey'], transform=ax_strain.transAxes)

    ax_strain.set_xticks(x)
    ax_strain.set_xticklabels(labels, fontsize=9)
    _style_axis(ax_strain, ylabel='Microstrain \u03b5 (%)')

    # Add headroom for annotations
    if has_strain:
        strain_max = max(s for s in strains if not np.isnan(s))
        ax_strain.set_ylim(top=strain_max * 1.35)

    title = f'Method Comparison \u2014 {filename}' if filename else 'Method Comparison'
    fig.suptitle(title, fontsize=11, fontweight='bold', y=0.98)
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fig.tight_layout(rect=[0, 0, 1, 0.95])
    return fig


# =============================================================================
# PLOT 8: MULTI-PEAK GALLERY
# =============================================================================

def plot_peak_gallery(fig: Figure,
                      fit_results: list,
                      analysed_peaks: list,
                      max_peaks: int = 12) -> Figure:
    """
    Gallery view: all fitted peaks in a grid of small subplots.

    Each subplot shows observed vs fitted with R² annotation.
    Quick visual QC of all fits at once.

    fit_results and analysed_peaks must be pre-matched (same length,
    corresponding entries). If lengths differ, pairs are matched by
    closest 2θ position.
    """
    fig.clear()

    # Match fits to peaks by position if lengths differ
    if len(fit_results) != len(analysed_peaks):
        matched = []
        for peak in analysed_peaks:
            best_fit = None
            best_dist = float('inf')
            for fit in fit_results:
                if fit.success and fit.data_x is not None:
                    dist = abs(fit.center_deg - peak.two_theta_deg)
                    if dist < best_dist:
                        best_dist = dist
                        best_fit = fit
            if best_fit is not None and best_dist < 1.0:
                matched.append((best_fit, peak))
    else:
        matched = [(f, p) for f, p in zip(fit_results, analysed_peaks)
                   if f.success and f.data_x is not None]

    if not matched:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, "No successful fits to display",
                ha='center', va='center', fontsize=14, color=C['grey'],
                transform=ax.transAxes)
        ax.set_axis_off()
        return fig

    n = min(len(matched), max_peaks)
    ncols = min(n, 4)
    nrows = int(np.ceil(n / ncols))

    gs = GridSpec(nrows, ncols, figure=fig, hspace=0.45, wspace=0.3)

    for idx, (fit, peak) in enumerate(matched[:n]):
        row = idx // ncols
        col = idx % ncols
        ax = fig.add_subplot(gs[row, col])

        # Data points
        ax.scatter(fit.data_x, fit.data_y, color=C['blue'], s=6, alpha=0.6,
                   edgecolors='none')

        # Fit curve
        ax.plot(fit.fit_x, fit.fit_y, color=C['vermillion'], linewidth=1.5)

        # Title
        ax.set_title(f'{fit.center_deg:.2f}\u00b0', fontsize=9, fontweight='bold',
                     pad=4)

        # Quality annotation — colour-coded
        r2_colour = C['green'] if fit.r_squared > 0.95 else (
                    C['orange'] if fit.r_squared > 0.85 else C['vermillion'])

        ax.text(0.95, 0.95, f'R\u00b2={fit.r_squared:.3f}\n\u03b2={fit.fwhm_deg:.3f}\u00b0',
                transform=ax.transAxes, fontsize=7, ha='right', va='top',
                color=r2_colour,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          alpha=0.7, edgecolor='none'))

        if peak.resolution_limited:
            ax.text(0.05, 0.95, '\u26a0 RL', transform=ax.transAxes,
                    fontsize=7, ha='left', va='top', color='red',
                    fontweight='bold')

        ax.tick_params(labelsize=6.5)
        ax.grid(True, alpha=0.15)

        # Axis labels on edges only
        if col == 0:
            ax.set_ylabel('Intensity', fontsize=7)
        else:
            ax.tick_params(labelleft=False)

        if row == nrows - 1:
            ax.set_xlabel('2\u03b8 (\u00b0)', fontsize=7)
        else:
            ax.tick_params(labelbottom=False)

    fig.suptitle('Peak Fit Gallery', fontsize=12, fontweight='bold')
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fig.tight_layout(rect=[0, 0, 1, 0.95])
    return fig

In [ ]:
"""
Volva 4.0 -- tkinter GUI
Notebook: Cell 1 = volva_engine, Cell 2 = volva_plots, Cell 3 = this
"""
import tkinter as tk
from tkinter import filedialog, messagebox, scrolledtext, ttk
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure
import os

try:
    import volva_engine as ve
except ImportError:
    class _S: pass
    ve = _S()
    for _n in ['WAVELENGTHS','SCHERRER_K','XRDLoader','BackgroundProcessor','PeakFitter',
               'InstrumentalCorrection','PeakProcessor','ScherrerAnalyzer','MonshiScherrerAnalyzer',
               'WilliamsonHallAnalyzer','HalderWagnerAnalyzer','SSPAnalyzer',
               'ResultsFormatter','ResultsExporter','run_all_methods']:
        if _n in globals(): setattr(ve, _n, globals()[_n])

try:
    import volva_plots as vp
except ImportError:
    class _P: pass
    vp = _P()
    for _n in ['plot_xrd_with_background','plot_standard_diagnostic','plot_xrd_with_fits',
               'plot_peak_inspector','plot_scherrer_vs_2theta','plot_linear_method',
               'plot_method_comparison','plot_peak_gallery']:
        if _n in globals(): setattr(vp, _n, globals()[_n])


class VolvaApp:
    VERSION = "4.0"

    def __init__(self, root):
        self.root = root
        self.root.title(f"Volva {self.VERSION} -- Crystallite Size & Strain Analysis")
        self.root.geometry("1500x950")
        self.root.minsize(1200, 750)

        self.sample_two_theta = None
        self.sample_intensity = None
        self.sample_filename = ""
        self.sample_bg = None
        self.sample_subtracted = None
        self.std_two_theta = None
        self.std_intensity = None
        self.std_filename = ""
        self.fitter = None
        self.inst_correction = ve.InstrumentalCorrection()
        self.analysed_peaks = []
        self.fit_results = []
        self.analysis_results = {}
        self._inspector_index = 0
        self._peak_use_flags = []  # Boolean flags for peak table

        # Settings (tuneable via Options > Settings)
        # Read defaults from engine constants where available, with fallbacks
        self.settings = {
            'min_r_squared': tk.StringVar(value=str(getattr(ve, 'DEFAULT_MIN_R_SQUARED', 0.65))),
            'min_snr': tk.StringVar(value=str(getattr(ve, 'DEFAULT_MIN_SNR', 5.0))),
            'sample_dedup_deg': tk.StringVar(value=str(getattr(ve, 'DEFAULT_SAMPLE_DEDUP_DEG', 0.3))),
            'std_dedup_deg': tk.StringVar(value=str(getattr(ve, 'DEFAULT_STD_DEDUP_DEG', 0.1))),
            'snip_iterations': tk.StringVar(value=str(getattr(ve, 'DEFAULT_SNIP_ITERATIONS', 40))),
        }

        self._build_menu()
        self._build_layout()
        self._build_controls()
        self._build_plot_area()
        self._build_results_area()
        self._init_plot()
        self._log("Volva 4.0 started. Load your XRD data to begin.")

    # ---- Menu ----
    def _build_menu(self):
        mb = tk.Menu(self.root)
        fm = tk.Menu(mb, tearoff=0)
        fm.add_command(label="Load Sample Data...", command=self._load_sample)
        fm.add_command(label="Load Standard (LaB6)...", command=self._load_standard)
        fm.add_separator()
        fm.add_command(label="Export Peak Data (CSV)...", command=self._export_peaks_csv)
        fm.add_command(label="Export Summary (CSV)...", command=self._export_summary_csv)
        fm.add_command(label="Export Figure...", command=self._export_figure)
        fm.add_separator()
        fm.add_command(label="Exit", command=self.root.quit)
        mb.add_cascade(label="File", menu=fm)

        am = tk.Menu(mb, tearoff=0)
        am.add_command(label="Detect & Fit Peaks", command=self._run_peak_processing)
        am.add_separator()
        for lbl, key in [("Scherrer",'Scherrer'),("Monshi-Scherrer",'Monshi-Scherrer'),
                         ("Williamson-Hall",'Williamson-Hall'),("Halder-Wagner",'Halder-Wagner'),("SSP",'SSP')]:
            am.add_command(label=f"Run {lbl}", command=lambda m=key: self._run_single_method(m))
        am.add_separator()
        am.add_command(label="Run All Methods", command=self._run_all)
        mb.add_cascade(label="Analysis", menu=am)

        vm = tk.Menu(mb, tearoff=0)
        for lbl, key in [("XRD + Background",'xrd_bg'),("XRD + Fits",'xrd_fits'),("LaB6 Diagnostic",'lab6'),
                         ("Peak Gallery",'gallery'),("Scherrer D vs 2t",'scherrer_d')]:
            vm.add_command(label=lbl, command=lambda k=key: self._switch_plot(k))
        vm.add_separator()
        for lbl, key in [("Williamson-Hall",'wh'),("Halder-Wagner",'hw'),("SSP",'ssp'),
                         ("Monshi-Scherrer",'monshi'),("Method Comparison",'comparison')]:
            vm.add_command(label=lbl, command=lambda k=key: self._switch_plot(k))
        mb.add_cascade(label="View", menu=vm)

        om = tk.Menu(mb, tearoff=0)
        om.add_command(label="Settings...", command=self._show_settings)
        mb.add_cascade(label="Options", menu=om)

        self.root.config(menu=mb)

    # ---- Layout ----
    def _build_layout(self):
        self.paned = tk.PanedWindow(self.root, orient=tk.HORIZONTAL, sashwidth=5)
        self.paned.pack(fill=tk.BOTH, expand=True, padx=2, pady=2)
        self.left_frame = tk.Frame(self.paned, width=380)
        self.paned.add(self.left_frame, minsize=320, stretch="never")
        self.right_frame = tk.Frame(self.paned)
        self.paned.add(self.right_frame, stretch="always")

    # ---- Controls ----
    def _build_controls(self):
        ctrl = tk.Frame(self.left_frame)
        ctrl.pack(fill=tk.BOTH, expand=False)

        # 1. Data
        f1 = tk.LabelFrame(ctrl, text="1. Data", padx=8, pady=5)
        f1.pack(fill=tk.X, padx=5, pady=(5,2))
        tk.Button(f1, text="Load Sample Data", command=self._load_sample).pack(fill=tk.X, pady=1)
        self.sample_label = tk.Label(f1, text="No sample loaded", fg="grey", wraplength=340, anchor='w')
        self.sample_label.pack(fill=tk.X, pady=1)
        tk.Button(f1, text="Load Standard (LaB6)", command=self._load_standard).pack(fill=tk.X, pady=1)
        self.std_label = tk.Label(f1, text="No standard loaded", fg="grey", wraplength=340, anchor='w')
        self.std_label.pack(fill=tk.X, pady=1)

        # 2. Wavelength
        f2 = tk.LabelFrame(ctrl, text="2. Wavelength", padx=8, pady=5)
        f2.pack(fill=tk.X, padx=5, pady=2)
        self.wavelength_var = tk.StringVar(value="Cu Ka")
        tk.Label(f2, text="Source:").grid(row=0, column=0, sticky='w')
        cb = ttk.Combobox(f2, textvariable=self.wavelength_var,
                          values=list(ve.WAVELENGTHS.keys())+["Custom"], state="readonly", width=18)
        cb.grid(row=0, column=1, padx=5, sticky='ew')
        cb.bind('<<ComboboxSelected>>', self._on_wl_change)
        tk.Label(f2, text="Custom (nm):").grid(row=1, column=0, sticky='w')
        self.custom_wl_var = tk.StringVar(value="0.154056")
        self.custom_wl_entry = tk.Entry(f2, textvariable=self.custom_wl_var, width=12, state='disabled')
        self.custom_wl_entry.grid(row=1, column=1, padx=5, sticky='w')
        self.doublet_var = tk.BooleanVar(value=True)
        tk.Checkbutton(f2, text="Fit Ka1/Ka2 doublet", variable=self.doublet_var).grid(
            row=2, column=0, columnspan=2, sticky='w')
        f2.columnconfigure(1, weight=1)

        # 3. Background
        f3 = tk.LabelFrame(ctrl, text="3. Background", padx=8, pady=5)
        f3.pack(fill=tk.X, padx=5, pady=2)
        self.bg_method_var = tk.StringVar(value="snip")
        tk.Label(f3, text="Method:").grid(row=0, column=0, sticky='w')
        ttk.Combobox(f3, textvariable=self.bg_method_var, values=["snip","rolling_minimum"],
                     state="readonly", width=18).grid(row=0, column=1, padx=5, sticky='ew')
        f3.columnconfigure(1, weight=1)

        # 4. Peak Detection
        f4 = tk.LabelFrame(ctrl, text="4. Peak Detection & Fitting", padx=8, pady=5)
        f4.pack(fill=tk.X, padx=5, pady=2)
        for i, (lbl, vname, default) in enumerate([
            ("Prominence:", "prominence_var", "0.05"),
            ("Min width (deg):", "min_width_var", "0.05"),
            ("Fit window max (deg):", "fit_window_var", "2.5"),
            ("Std min R2:", "std_min_rsq_var", "0.7"),
        ]):
            tk.Label(f4, text=lbl).grid(row=i, column=0, sticky='w', pady=1)
            v = tk.StringVar(value=default)
            setattr(self, vname, v)
            tk.Entry(f4, textvariable=v, width=8).grid(row=i, column=1, padx=5, sticky='w', pady=1)
        f4.columnconfigure(1, weight=1)

        # 5. Scherrer K
        f5 = tk.LabelFrame(ctrl, text="5. Scherrer Constant (K)", padx=8, pady=5)
        f5.pack(fill=tk.X, padx=5, pady=2)
        self.scherrer_k_var = tk.StringVar(value="Spherical (FWHM)")
        tk.Label(f5, text="Shape:").grid(row=0, column=0, sticky='w')
        k_cb = ttk.Combobox(f5, textvariable=self.scherrer_k_var,
                            values=list(ve.SCHERRER_K.keys())+["Custom"], state="readonly", width=22)
        k_cb.grid(row=0, column=1, padx=5, sticky='ew')
        k_cb.bind('<<ComboboxSelected>>', self._on_k_change)
        tk.Label(f5, text="Custom K:").grid(row=1, column=0, sticky='w')
        self.custom_k_var = tk.StringVar(value="0.94")
        self.custom_k_entry = tk.Entry(f5, textvariable=self.custom_k_var, width=8, state='disabled')
        self.custom_k_entry.grid(row=1, column=1, padx=5, sticky='w')
        f5.columnconfigure(1, weight=1)

        # 6. Run
        f6 = tk.LabelFrame(ctrl, text="6. Run Analysis", padx=8, pady=5)
        f6.pack(fill=tk.X, padx=5, pady=2)
        tk.Button(f6, text="Detect & Fit Peaks", command=self._run_peak_processing,
                  bg='#4CAF50', fg='white', font=('',9,'bold')).pack(fill=tk.X, pady=2)
        bf = tk.Frame(f6); bf.pack(fill=tk.X, pady=2)
        for i, (t, m) in enumerate([("Scherrer",'Scherrer'),("Monshi",'Monshi-Scherrer'),
                                     ("W-H",'Williamson-Hall'),("H-W",'Halder-Wagner'),("SSP",'SSP')]):
            tk.Button(bf, text=t, width=6, command=lambda m=m: self._run_single_method(m)).grid(
                row=0, column=i, padx=1, pady=1)
        tk.Button(f6, text="Run All Methods", command=self._run_all,
                  bg='#2196F3', fg='white', font=('',9,'bold')).pack(fill=tk.X, pady=2)

        # Status
        self.status_var = tk.StringVar(value="Ready")
        tk.Label(ctrl, textvariable=self.status_var, relief=tk.SUNKEN, anchor='w', fg='#333').pack(
            fill=tk.X, padx=5, pady=(2,0), side=tk.BOTTOM)

    # ---- Plot area ----
    def _build_plot_area(self):
        # 1. Single row of buttons at bottom (packed first = guaranteed visible)
        bf = tk.Frame(self.right_frame)
        bf.pack(side=tk.BOTTOM, fill=tk.X, pady=(2, 4))

        all_buttons = [
            ("XRD+BG",'xrd_bg'), ("Fits",'xrd_fits'), ("LaB6",'lab6'),
            ("Gallery",'gallery'), ("D vs 2\u03b8",'scherrer_d'),
            ("W-H",'wh'), ("H-W",'hw'), ("SSP",'ssp'), ("Monshi",'monshi'),
            ("Compare",'comparison'), ("Results",'results'), ("Log",'log'),
        ]
        for t, k in all_buttons:
            btn = tk.Button(bf, text=t, width=7, command=lambda k=k: self._switch_plot(k))
            btn.pack(side=tk.LEFT, padx=1)
            if k == 'results':
                self._results_btn = btn

        # Peak inspector nav
        tk.Label(bf, text="  \u2502", fg='#999').pack(side=tk.LEFT)
        tk.Button(bf, text="\u25c0", width=2, command=self._prev_peak).pack(side=tk.LEFT, padx=1)
        tk.Button(bf, text="Inspect", width=6,
                  command=lambda: self._switch_plot('inspector')).pack(side=tk.LEFT, padx=1)
        tk.Button(bf, text="\u25b6", width=2, command=self._next_peak).pack(side=tk.LEFT, padx=1)
        self.inspector_label = tk.Label(bf, text="", fg='#333', font=('', 8))
        self.inspector_label.pack(side=tk.LEFT, padx=4)

        # 2. Toolbar above buttons
        self.toolbar_frame = tk.Frame(self.right_frame)
        self.toolbar_frame.pack(side=tk.BOTTOM, fill=tk.X)

        # 3. View container fills everything else
        self.view_container = tk.Frame(self.right_frame)
        self.view_container.pack(side=tk.TOP, fill=tk.BOTH, expand=True)

        # Create matplotlib canvas
        self.fig = Figure(figsize=(9, 5.5), dpi=100)
        self.fig.set_facecolor('white')
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.view_container)
        self.canvas_widget = self.canvas.get_tk_widget()
        self.canvas_widget.configure(height=1)

        self.toolbar = NavigationToolbar2Tk(self.canvas, self.toolbar_frame)
        self.toolbar.update()

        self.results_text = scrolledtext.ScrolledText(
            self.view_container, wrap=tk.WORD, font=('Consolas', 9))
        self.results_text.config(state='disabled')

        self.log_text = scrolledtext.ScrolledText(
            self.view_container, wrap=tk.WORD, font=('Consolas', 8))
        self.log_text.config(state='disabled')

        # Start with canvas visible
        self.canvas_widget.pack(fill=tk.BOTH, expand=True)
        self._active_view_widget = self.canvas_widget

    # ---- Results area with peak table ----
    def _build_results_area(self):
        # --- Peak table stays on the left ---
        pk_frame = tk.LabelFrame(self.left_frame, text="Peaks", padx=4, pady=4)
        pk_frame.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)

        pk_btn = tk.Frame(pk_frame)
        pk_btn.pack(fill=tk.X, padx=2, pady=2)
        tk.Button(pk_btn, text="Select All", width=8, command=self._select_all_peaks).pack(side=tk.LEFT, padx=2)
        tk.Button(pk_btn, text="Deselect All", width=8, command=self._deselect_all_peaks).pack(side=tk.LEFT, padx=2)
        tk.Button(pk_btn, text="Re-run", width=8, command=self._rerun_with_selection,
                  bg='#2196F3', fg='white').pack(side=tk.RIGHT, padx=2)

        cols = ('use', '2theta', 'fwhm_obs', 'beta_corr', 'D_nm', 'R2', 'flag')
        self.peak_tree = ttk.Treeview(pk_frame, columns=cols, show='headings', height=6, selectmode='browse')
        for col, heading, width in [
            ('use', 'Use', 32), ('2theta', '2\u03b8', 55), ('fwhm_obs', '\u03b2_obs', 50),
            ('beta_corr', '\u03b2_corr', 50), ('D_nm', 'D(nm)', 45),
            ('R2', 'R\u00b2', 42), ('flag', '', 25),
        ]:
            self.peak_tree.heading(col, text=heading)
            anchor = 'center' if col in ('use', 'flag') else 'e'
            self.peak_tree.column(col, width=width, anchor=anchor, stretch=False)

        self.peak_tree.pack(fill=tk.BOTH, expand=True, padx=2, pady=2)
        self.peak_tree.bind('<ButtonRelease-1>', self._on_peak_click)

    def _init_plot(self):
        self.fig.clear()
        ax = self.fig.add_subplot(111)
        ax.text(0.5, 0.5, 'Load XRD data to begin', ha='center', va='center',
                fontsize=14, color='#999', transform=ax.transAxes)
        ax.set_axis_off()
        self.canvas.draw_idle()

    # ==== Peak table methods ====

    def _populate_peak_table(self):
        """Fill the peak table from current analysed_peaks."""
        self.peak_tree.delete(*self.peak_tree.get_children())
        self._peak_use_flags = [True] * len(self.analysed_peaks)

        k = self._get_k_constant() or 0.94
        _, wl_nm = self._get_wavelength_info()
        if wl_nm is None:
            wl_nm = 0.154056

        for i, p in enumerate(self.analysed_peaks):
            cos_t = np.cos(p.theta_rad)
            if p.beta_corrected_rad > 0 and cos_t > 0:
                d_nm = k * wl_nm / (p.beta_corrected_rad * cos_t)
            else:
                d_nm = float('nan')

            flag = "RL" if p.resolution_limited else ""
            use_str = "\u2713"  # tick mark

            self.peak_tree.insert('', 'end', iid=str(i), values=(
                use_str,
                f"{p.two_theta_deg:.3f}",
                f"{p.fwhm_obs_deg:.4f}",
                f"{p.beta_corrected_deg:.4f}",
                f"{d_nm:.1f}" if not np.isnan(d_nm) else "N/A",
                f"{p.fit_quality:.3f}",
                flag,
            ))

    def _on_peak_click(self, event):
        """Toggle peak use flag when clicking the Use column, or inspect on double-click."""
        region = self.peak_tree.identify_region(event.x, event.y)
        if region != 'cell':
            return
        col = self.peak_tree.identify_column(event.x)
        item = self.peak_tree.identify_row(event.y)
        if not item:
            return

        idx = int(item)

        if col == '#1':  # 'use' column
            self._peak_use_flags[idx] = not self._peak_use_flags[idx]
            use_str = "\u2713" if self._peak_use_flags[idx] else "\u2717"
            vals = list(self.peak_tree.item(item, 'values'))
            vals[0] = use_str
            self.peak_tree.item(item, values=vals)
        else:
            # Click on any other column: navigate inspector to this peak
            self._inspector_index = idx
            self._switch_plot('inspector')

    def _select_all_peaks(self):
        """Mark all peaks as used."""
        self._peak_use_flags = [True] * len(self.analysed_peaks)
        for item in self.peak_tree.get_children():
            vals = list(self.peak_tree.item(item, 'values'))
            vals[0] = "\u2713"
            self.peak_tree.item(item, values=vals)

    def _deselect_all_peaks(self):
        """Mark all peaks as unused."""
        self._peak_use_flags = [False] * len(self.analysed_peaks)
        for item in self.peak_tree.get_children():
            vals = list(self.peak_tree.item(item, 'values'))
            vals[0] = "\u2717"
            self.peak_tree.item(item, values=vals)

    def _get_selected_peaks(self):
        """Return only peaks flagged as used."""
        return [p for p, use in zip(self.analysed_peaks, self._peak_use_flags) if use]

    def _rerun_with_selection(self):
        """Re-run all methods using only selected peaks."""
        selected = self._get_selected_peaks()
        if not selected:
            messagebox.showwarning("No Peaks", "No peaks selected for analysis.")
            return
        _, wl_nm = self._get_wavelength_info()
        k = self._get_k_constant()
        if wl_nm is None or k is None:
            return
        self.status_var.set(f"Re-running with {len(selected)}/{len(self.analysed_peaks)} peaks...")
        self.root.update()
        self.analysis_results = ve.run_all_methods(selected, wl_nm, k)
        fmts = {'Scherrer': ve.ResultsFormatter.format_scherrer,
                'Monshi-Scherrer': ve.ResultsFormatter.format_monshi_scherrer,
                'Williamson-Hall': ve.ResultsFormatter.format_williamson_hall,
                'Halder-Wagner': ve.ResultsFormatter.format_halder_wagner,
                'SSP': ve.ResultsFormatter.format_ssp}
        lines = []
        for n, r in self.analysis_results.items():
            f = fmts.get(n)
            if f:
                lines.append(f(r))
                lines.append("")
        lines.append(ve.ResultsFormatter.format_comparison(self.analysis_results))
        self._set_results("\n".join(lines))
        self.status_var.set(f"Analysis complete ({len(selected)} peaks).")
        self._switch_plot('comparison')

    # ==== Parameter getters ====

    def _get_wavelength_info(self):
        n = self.wavelength_var.get()
        if n == "Custom":
            try:
                ka1 = float(self.custom_wl_var.get())
            except ValueError:
                messagebox.showerror("Error", "Invalid wavelength.")
                return None, None
            return {'ka1': ka1, 'ka2': ka1, 'ratio': 0.0}, ka1
        wl = ve.WAVELENGTHS[n]
        return wl, wl['ka1']

    def _get_k_constant(self):
        n = self.scherrer_k_var.get()
        if n == "Custom":
            try:
                return float(self.custom_k_var.get())
            except ValueError:
                messagebox.showerror("Error", "Invalid K.")
                return None
        return ve.SCHERRER_K.get(n, 0.94)

    def _get_float(self, var, name, positive=True):
        try:
            v = float(var.get())
            if positive and v <= 0:
                messagebox.showwarning("Warning", f"{name} must be positive.")
                return None
            return v
        except ValueError:
            messagebox.showerror("Error", f"Invalid {name}.")
            return None

    # ==== Callbacks ====

    def _on_wl_change(self, e=None):
        if self.wavelength_var.get() == "Custom":
            self.custom_wl_entry.config(state='normal')
            self.doublet_var.set(False)
        else:
            self.custom_wl_entry.config(state='disabled')
            self.custom_wl_var.set(str(ve.WAVELENGTHS[self.wavelength_var.get()]['ka1']))
            self.doublet_var.set(True)

    def _on_k_change(self, e=None):
        if self.scherrer_k_var.get() == "Custom":
            self.custom_k_entry.config(state='normal')
        else:
            self.custom_k_entry.config(state='disabled')
            self.custom_k_var.set(str(ve.SCHERRER_K.get(self.scherrer_k_var.get(), 0.94)))

    def _prev_peak(self):
        if self.analysed_peaks:
            self._inspector_index = max(0, self._inspector_index - 1)
            self._switch_plot('inspector')

    def _next_peak(self):
        if self.analysed_peaks:
            self._inspector_index = min(len(self.analysed_peaks) - 1, self._inspector_index + 1)
            self._switch_plot('inspector')

    # ==== Settings dialog ====

    def _show_settings(self):
        """Show the Settings dialog for tuneable engine parameters."""
        dlg = tk.Toplevel(self.root)
        dlg.title("Volva Settings")
        dlg.geometry("380x320")
        dlg.resizable(False, False)
        dlg.transient(self.root)
        dlg.grab_set()

        frame = tk.LabelFrame(dlg, text="Engine Parameters", padx=12, pady=8)
        frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        fields = [
            ("Min fit R²:", self.settings['min_r_squared'],
             "Minimum R² to accept a peak fit"),
            ("Min SNR:", self.settings['min_snr'],
             "Minimum signal-to-noise ratio to attempt fitting"),
            ("Sample dedup (°):", self.settings['sample_dedup_deg'],
             "Minimum separation between distinct sample peaks"),
            ("Standard dedup (°):", self.settings['std_dedup_deg'],
             "Minimum separation between distinct standard peaks"),
            ("SNIP iterations:", self.settings['snip_iterations'],
             "Background estimation iterations (more = smoother)"),
        ]

        for i, (label, var, tooltip) in enumerate(fields):
            tk.Label(frame, text=label, anchor='w').grid(
                row=i, column=0, sticky='w', pady=4)
            entry = tk.Entry(frame, textvariable=var, width=10)
            entry.grid(row=i, column=1, padx=8, pady=4, sticky='w')
            tk.Label(frame, text=tooltip, fg='#666', font=('', 8),
                     anchor='w', wraplength=160).grid(
                row=i, column=2, sticky='w', padx=(0, 4))

        frame.columnconfigure(2, weight=1)

        # Buttons
        btn_frame = tk.Frame(dlg)
        btn_frame.pack(fill=tk.X, padx=10, pady=(0, 10))

        def _reset_defaults():
            self.settings['min_r_squared'].set(str(getattr(ve, 'DEFAULT_MIN_R_SQUARED', 0.65)))
            self.settings['min_snr'].set(str(getattr(ve, 'DEFAULT_MIN_SNR', 5.0)))
            self.settings['sample_dedup_deg'].set(str(getattr(ve, 'DEFAULT_SAMPLE_DEDUP_DEG', 0.3)))
            self.settings['std_dedup_deg'].set(str(getattr(ve, 'DEFAULT_STD_DEDUP_DEG', 0.1)))
            self.settings['snip_iterations'].set(str(getattr(ve, 'DEFAULT_SNIP_ITERATIONS', 40)))

        tk.Button(btn_frame, text="Reset Defaults", command=_reset_defaults).pack(
            side=tk.LEFT, padx=4)
        tk.Button(btn_frame, text="Close", command=dlg.destroy).pack(
            side=tk.RIGHT, padx=4)

        # Log current values
        self._log("Settings dialog opened. Changes apply on next analysis run.")

    def _get_setting(self, key, cast=float):
        """Safely retrieve a setting value with type casting."""
        try:
            return cast(self.settings[key].get())
        except (ValueError, KeyError):
            defaults = {
                'min_r_squared': 0.8, 'min_snr': 5.0,
                'sample_dedup_deg': 0.15, 'std_dedup_deg': 0.1,
                'snip_iterations': 40,
            }
            return defaults.get(key, 0)

    def _subtract_background(self):
        """Background-subtract the sample data using current settings."""
        method = self.bg_method_var.get()
        kwargs = {}
        if method == 'snip':
            kwargs['n_iterations'] = self._get_setting('snip_iterations', cast=int)
        self.sample_subtracted, self.sample_bg = ve.BackgroundProcessor.subtract(
            self.sample_two_theta, self.sample_intensity, method=method, **kwargs)

    # ==== Data loading ====

    def _load_sample(self):
        fp = filedialog.askopenfilename(title="Select Sample XRD Data",
                                         filetypes=[("XRD Data","*.xy *.txt *.dat *.csv"),("All","*.*")])
        if not fp:
            return
        tt, ii, msg = ve.XRDLoader.load(fp)
        if tt is not None:
            self.sample_two_theta = tt
            self.sample_intensity = ii
            self.sample_filename = os.path.basename(fp)
            self._subtract_background()
            self.analysed_peaks = []
            self.fit_results = []
            self.analysis_results = {}
            self._peak_use_flags = []
            self._results_btn.config(bg='SystemButtonFace', fg='black',
                                     activebackground='SystemButtonFace',
                                     activeforeground='black')
            self.sample_label.config(text=f">> {self.sample_filename} ({len(tt)} pts)", fg="darkgreen")
            self._log(f"Sample loaded: {msg}")
            self.status_var.set(f"Sample: {self.sample_filename}")
            self._switch_plot('xrd_bg')
        else:
            messagebox.showerror("Error", msg)

    def _load_standard(self):
        fp = filedialog.askopenfilename(title="Select Standard (LaB6) XRD Data",
                                         filetypes=[("XRD Data","*.xy *.txt *.dat *.csv"),("All","*.*")])
        if not fp:
            return
        tt, ii, msg = ve.XRDLoader.load(fp)
        if tt is not None:
            self.std_two_theta = tt
            self.std_intensity = ii
            self.std_filename = os.path.basename(fp)
            self.std_label.config(text=f">> {self.std_filename} ({len(tt)} pts)", fg="darkgreen")
            self._log(f"Standard loaded: {msg}")
            self._process_standard()
            self._switch_plot('lab6')
        else:
            messagebox.showerror("Error", msg)

    def _process_standard(self):
        wl, _ = self._get_wavelength_info()
        if wl is None:
            return
        self.fitter = ve.PeakFitter(wl)
        prom = self._get_float(self.prominence_var, "Prominence") or 0.05
        mw = self._get_float(self.min_width_var, "Min width") or 0.05
        fw = self._get_float(self.fit_window_var, "Fit window") or 1.0
        mr = self._get_float(self.std_min_rsq_var, "Std min R2") or 0.7
        self.status_var.set("Processing standard...")
        self.root.update()
        self.inst_correction = ve.InstrumentalCorrection()
        msg = self.inst_correction.process_standard(
            self.std_two_theta, self.std_intensity, self.fitter,
            bg_method=self.bg_method_var.get(), prominence_factor=prom,
            min_width_deg=mw, fit_window_deg=fw,
            enable_doublet=self.doublet_var.get(), min_r_squared=mr,
            dedup_deg=self._get_setting('std_dedup_deg'))
        self._log(msg)
        n = len(self.inst_correction.standard_peaks)
        if n > 0:
            self.std_label.config(text=f">> {self.std_filename} ({n} peaks)", fg="darkgreen")
            self.status_var.set(f"Standard: {n} peaks")
        else:
            self.std_label.config(text=f"!! {self.std_filename} (0 peaks)", fg="orange")
            self.status_var.set("Warning: no standard peaks accepted")

    # ==== Analysis ====

    def _run_peak_processing(self):
        if self.sample_two_theta is None:
            messagebox.showwarning("Missing Data", "Load sample first.")
            return
        wl, wl_nm = self._get_wavelength_info()
        if wl is None:
            return
        self.fitter = ve.PeakFitter(wl)
        prom = self._get_float(self.prominence_var, "Prominence")
        mw = self._get_float(self.min_width_var, "Min width")
        fw = self._get_float(self.fit_window_var, "Fit window")
        if any(v is None for v in [prom, mw, fw]):
            return
        if self.std_two_theta is not None and not self.inst_correction.is_loaded:
            self._process_standard()
        self._subtract_background()
        self.status_var.set("Detecting and fitting peaks...")
        self.root.update()
        inst = self.inst_correction if self.inst_correction.is_loaded else None
        self.analysed_peaks, self.fit_results, log = ve.PeakProcessor.process_sample(
            self.sample_two_theta, self.sample_intensity, self.fitter,
            inst_correction=inst, wavelength_nm=wl_nm,
            bg_method=self.bg_method_var.get(), prominence_factor=prom,
            min_width_deg=mw, fit_window_deg=fw,
            enable_doublet=self.doublet_var.get(),
            min_r_squared=self._get_setting('min_r_squared'),
            min_snr=self._get_setting('min_snr'),
            dedup_deg=self._get_setting('sample_dedup_deg'),
            snip_iterations=self._get_setting('snip_iterations', cast=int))
        self._log(log)
        self.analysis_results = {}
        self._inspector_index = 0
        n = len(self.analysed_peaks)
        if n > 0:
            self.status_var.set(f"Peak processing: {n} peaks")
            self._populate_peak_table()
            self._switch_plot('xrd_fits')
        else:
            self.status_var.set("No peaks fitted.")
            messagebox.showinfo("Peaks", "No peaks detected/fitted. Try adjusting parameters.")

    def _run_single_method(self, method_name):
        peaks = self._get_selected_peaks() if self._peak_use_flags else self.analysed_peaks
        if not peaks:
            messagebox.showwarning("Missing", "Detect peaks first.")
            return
        _, wl_nm = self._get_wavelength_info()
        k = self._get_k_constant()
        if wl_nm is None or k is None:
            return
        self.status_var.set(f"Running {method_name}...")
        self.root.update()
        analysers = {
            'Scherrer': (ve.ScherrerAnalyzer, ve.ResultsFormatter.format_scherrer),
            'Monshi-Scherrer': (ve.MonshiScherrerAnalyzer, ve.ResultsFormatter.format_monshi_scherrer),
            'Williamson-Hall': (ve.WilliamsonHallAnalyzer, ve.ResultsFormatter.format_williamson_hall),
            'Halder-Wagner': (ve.HalderWagnerAnalyzer, ve.ResultsFormatter.format_halder_wagner),
            'SSP': (ve.SSPAnalyzer, ve.ResultsFormatter.format_ssp),
        }
        cls, fmt = analysers[method_name]
        result = cls(wl_nm).analyse(peaks, k)
        self.analysis_results[method_name] = result
        self._set_results(fmt(result))
        self.status_var.set(f"{method_name} complete.")
        pm = {'Scherrer':'scherrer_d','Williamson-Hall':'wh','Halder-Wagner':'hw',
              'SSP':'ssp','Monshi-Scherrer':'monshi'}
        self._switch_plot(pm.get(method_name, 'xrd_fits'))

    def _run_all(self):
        peaks = self._get_selected_peaks() if self._peak_use_flags else self.analysed_peaks
        if not peaks:
            messagebox.showwarning("Missing", "Detect peaks first.")
            return
        _, wl_nm = self._get_wavelength_info()
        k = self._get_k_constant()
        if wl_nm is None or k is None:
            return
        self.status_var.set("Running all methods...")
        self.root.update()
        self.analysis_results = ve.run_all_methods(peaks, wl_nm, k)
        fmts = {'Scherrer': ve.ResultsFormatter.format_scherrer,
                'Monshi-Scherrer': ve.ResultsFormatter.format_monshi_scherrer,
                'Williamson-Hall': ve.ResultsFormatter.format_williamson_hall,
                'Halder-Wagner': ve.ResultsFormatter.format_halder_wagner,
                'SSP': ve.ResultsFormatter.format_ssp}
        lines = []
        for n, r in self.analysis_results.items():
            f = fmts.get(n)
            if f:
                lines.append(f(r))
                lines.append("")
        lines.append(ve.ResultsFormatter.format_comparison(self.analysis_results))
        self._set_results("\n".join(lines))
        self.status_var.set("All methods complete.")
        self._switch_plot('comparison')

    # ==== Plotting (delegates to volva_plots) ====

    def _show_widget(self, widget):
        """Swap the visible widget in the view container (canvas, results, or log)."""
        if self._active_view_widget is widget:
            return
        self._active_view_widget.pack_forget()
        widget.pack(in_=self.view_container, fill=tk.BOTH, expand=True)
        self._active_view_widget = widget

    def _switch_plot(self, view):
        if view == 'results':
            self._show_widget(self.results_text)
            return
        elif view == 'log':
            self._show_widget(self.log_text)
            return

        # All other views use the canvas
        self._show_widget(self.canvas_widget)

        dispatch = {
            'xrd_bg': self._plot_xrd_bg,
            'xrd_fits': self._plot_xrd_fits,
            'lab6': self._plot_lab6,
            'gallery': self._plot_gallery,
            'inspector': self._plot_inspector,
            'scherrer_d': self._plot_scherrer_d,
            'wh': lambda: self._plot_linear('Williamson-Hall'),
            'hw': lambda: self._plot_linear('Halder-Wagner'),
            'ssp': lambda: self._plot_linear('SSP'),
            'monshi': lambda: self._plot_linear('Monshi-Scherrer'),
            'comparison': self._plot_comparison,
        }
        fn = dispatch.get(view)
        if fn:
            fn()

    def _plot_xrd_bg(self):
        if self.sample_two_theta is None:
            return
        if self.sample_bg is None:
            self._subtract_background()
        vp.plot_xrd_with_background(self.fig, self.sample_two_theta, self.sample_intensity,
                                     self.sample_bg, self.sample_subtracted,
                                     filename=self.sample_filename,
                                     std_two_theta=self.std_two_theta,
                                     std_intensity=self.std_intensity,
                                     std_filename=self.std_filename)
        self.canvas.draw_idle()

    def _plot_xrd_fits(self):
        if self.sample_two_theta is None:
            return
        if self.sample_bg is None:
            self._subtract_background()
        inst = self.inst_correction if self.inst_correction.is_loaded else None
        vp.plot_xrd_with_fits(self.fig, self.sample_two_theta, self.sample_intensity,
                               self.sample_bg, self.fit_results, self.analysed_peaks,
                               filename=self.sample_filename, inst_correction=inst)
        self.canvas.draw_idle()

    def _plot_lab6(self):
        if self.std_two_theta is None or not self.inst_correction.is_loaded:
            self._show_placeholder("Load and process a LaB6 standard first")
            return
        vp.plot_standard_diagnostic(self.fig, self.std_two_theta, self.std_intensity,
                                     self.inst_correction, filename=self.std_filename)
        self.canvas.draw_idle()

    def _plot_gallery(self):
        if not self.analysed_peaks:
            self._show_placeholder("Detect and fit peaks first")
            return
        vp.plot_peak_gallery(self.fig, self.fit_results, self.analysed_peaks)
        self.canvas.draw_idle()

    def _plot_inspector(self):
        if not self.analysed_peaks:
            return
        idx = self._inspector_index
        peak = self.analysed_peaks[idx]
        self.inspector_label.config(
            text=f"Peak {idx+1}/{len(self.analysed_peaks)}: {peak.two_theta_deg:.3f} deg")
        matched_fit = None
        for f in self.fit_results:
            if f.success and abs(f.center_deg - peak.two_theta_deg) < 0.5:
                matched_fit = f
                break
        if matched_fit is None:
            return
        wl, _ = self._get_wavelength_info()
        vp.plot_peak_inspector(self.fig, matched_fit, peak,
                                peak_index=idx, wavelength_info=wl)
        self.canvas.draw_idle()

    def _plot_scherrer_d(self):
        r = self.analysis_results.get('Scherrer')
        if r is None:
            if self.analysed_peaks:
                self._run_single_method('Scherrer')
                return
            self._show_placeholder("Run Scherrer analysis first")
            return
        vp.plot_scherrer_vs_2theta(self.fig, r, filename=self.sample_filename)
        self.canvas.draw_idle()

    def _plot_linear(self, method_name):
        r = self.analysis_results.get(method_name)
        if r is None:
            if self.analysed_peaks:
                self._run_single_method(method_name)
                return
            self._show_placeholder(f"Run {method_name} first")
            return
        vp.plot_linear_method(self.fig, r, method_name)
        self.canvas.draw_idle()

    def _plot_comparison(self):
        if not self.analysis_results:
            if self.analysed_peaks:
                self._run_all()
                return
            self._show_placeholder("Run analysis methods first")
            return
        vp.plot_method_comparison(self.fig, self.analysis_results,
                                   filename=self.sample_filename)
        self.canvas.draw_idle()

    def _show_placeholder(self, msg):
        self.fig.clear()
        ax = self.fig.add_subplot(111)
        ax.text(0.5, 0.5, msg, ha='center', va='center', fontsize=14,
                color='#999', transform=ax.transAxes)
        ax.set_axis_off()
        self.canvas.draw_idle()

    # ==== Export ====

    def _export_peaks_csv(self):
        if not self.analysed_peaks:
            messagebox.showwarning("No Data", "No peaks.")
            return
        fp = filedialog.asksaveasfilename(defaultextension=".csv",
                                           filetypes=[("CSV","*.csv")],
                                           initialfile="volva_peak_data.csv")
        if fp:
            ve.ResultsExporter.export_peak_data(self.analysed_peaks, fp)
            self._log(f"Exported: {fp}")

    def _export_summary_csv(self):
        if not self.analysis_results:
            messagebox.showwarning("No Data", "No results.")
            return
        fp = filedialog.asksaveasfilename(defaultextension=".csv",
                                           filetypes=[("CSV","*.csv")],
                                           initialfile="volva_summary.csv")
        if fp:
            ve.ResultsExporter.export_summary(self.analysis_results, fp)
            self._log(f"Exported: {fp}")

    def _export_figure(self):
        fp = filedialog.asksaveasfilename(defaultextension=".png",
                                           filetypes=[("PNG","*.png"),("PDF","*.pdf"),
                                                      ("TIFF","*.tiff"),("SVG","*.svg")],
                                           initialfile="volva_figure.png")
        if fp:
            self.fig.savefig(fp, dpi=600, bbox_inches='tight', facecolor='white')
            self._log(f"Figure: {fp} (600 DPI)")

    # ==== Text helpers ====

    def _set_results(self, text):
        self.results_text.config(state='normal')
        self.results_text.delete(1.0, tk.END)
        self.results_text.insert(tk.END, text)
        self.results_text.config(state='disabled')
        self.results_text.see(1.0)
        self._results_btn.config(bg='#4CAF50', fg='white', activebackground='#388E3C',
                                 activeforeground='white')
        self._switch_plot('results')  # Show results view

    def _log(self, msg):
        self.log_text.config(state='normal')
        existing = self.log_text.get(1.0, tk.END).strip()
        self.log_text.insert(tk.END, ("\n" if existing else "") + msg)
        self.log_text.config(state='disabled')
        self.log_text.see(tk.END)

    def run(self):
        self.root.mainloop()


def launch():
    root = tk.Tk()
    app = VolvaApp(root)
    app.run()

if __name__ == "__main__":
    launch()